# pFedES on VeReMi NextGen — 20 clients

Proxy homogeneous feature-extractor sharing (Yi et al., *pFedES*), Eq. (4)–(11), with
**DAGSNet** (395,024 params) as every client's personalized classifier F_k and a
DAGSNet-type proxy extractor G (407,874 params, output dim 66) shared through the server,
on VeReMi NextGen (16 classes, 66 features).

| from the paper | from `knowledge/` and the owner's decisions |
|---|---|
| iterative training: freeze G → train F_k on [G(x); x] with μ-weighted CE; freeze F_k → train G | DAGSNet architecture, 66-feature order, class order, seed 42 |
| Eq. (11) aggregation of θ over all K = 20 clients (C = 100 %) | batch 512, 2 rounds × E = E_fe = 1, AdamW / wd 0.0001, lr 0.001 → 1e-05 cosine per round, μ = 0.5 |

**Evaluation:** every round, every client's F_k(x) on the full 10,761,343-row test set;
10 metrics per client, mean/std/min/max over all 20 clients. Every client trains in every round, so every confusion matrix is recomputed from scratch each round: the carry-forward cache for unselected clients is never used and `cache_mismatch` stays 0.

**Deviations, all deliberate:** proxy is DAGSNet-type not the paper's 2-conv CNN; AdamW
not SGD, with a per-round cosine learning-rate schedule (0.001 at round 1 →
1e-05 at round 2, constant within a round, shared by η_ω and η_θ) instead of
the paper's constant 0.01; μ and E_fe are the owner's values (the paper's appendix is
unavailable); Step 1 runs its two forwards as one concatenated batch (BatchNorm sees both
halves); Eq. (11) normalises over the participating set, which at C = 100 % is every client; C is 100 % in all three scenarios by the owner's decision, not the paper's 100 %/20 %/10 %. Per-round output is **weights only** (G + every F_k as
state_dict tensors); `proj/ckpt.py::load_weights` rebuilds them at any round, and the last
cell re-derives every published metric from the confusion matrices on disk.


In [ ]:
import os, subprocess, sys, time, torch
T0 = time.monotonic()     # session clock: the 12 h cap charges for spawn and compile too.
n = torch.cuda.device_count()
assert n == 2, f"expected 2 GPUs, got {n}. machine_shape must be NvidiaTeslaT4."
for i in range(n):
    cap = torch.cuda.get_device_capability(i)
    assert cap == (7, 5), f"GPU {i} is {cap}, expected (7,5) Tesla T4"
    print(i, torch.cuda.get_device_name(i), cap,
          f"{torch.cuda.get_device_properties(i).total_memory/2**30:.1f} GiB")
print("torch", torch.__version__, "| python", sys.version.split()[0])
# NCCL is unused (FL clients never form a process group) but P2P probing can still hang.
os.environ["NCCL_P2P_DISABLE"] = "1"; os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["TORCHINDUCTOR_COMPILE_THREADS"] = "1"
os.makedirs("/kaggle/working/proj", exist_ok=True)
open("/kaggle/working/proj/__init__.py", "w").close()
sys.path.insert(0, "/kaggle/working")


In [ ]:
CFG = dict(
    # --- architecture: knowledge/architecture.md, frozen (F_k 395,024 / G 407,874 params)
    patch_len=6, stem_ch=96, dense_growth=32, dense_layers=3,
    incep_modules=2, fire_modules=3, dropout=0.1,
    num_classes=16, n_features=66,
    # --- pFedES: owner's decisions 2026-09-11 (paper: SGD 0.01; mu, E_fe unspecified);
    #     LR schedule 2026-09-13: cosine per round lr -> lr_min (proj/pfedes.py::lr_at)
    lr=0.001, lr_schedule="cosine", lr_min=1e-05,
    weight_decay=0.0001, mu=0.5,
    local_epochs=1, proxy_epochs=1,
    participation=1.0,        # owner 2026-09-12: C = 100 % in all scenarios
    rounds=2, clip=1.0, seed=42,
    # --- compute: knowledge/dataset.md 4
    n_clients=20, batch=512, eval_batch=16384,
    device="cuda", world_size=2, compile=True,
    # Each client starts fresh GradScalers at 2**16 and spends a few steps calibrating.
    # Fixed before the first measurement so it cannot be widened afterwards.
    max_skips_per_client=16,
    # Clients not selected in a round carry their confusion matrix; everyone is re-evaluated
    # every eval_all_every rounds and at the last round, and the cache is checked then.
    eval_all_every=1000,
    preds_rounds=[],   # (N, 10.76 M) uint8 per listed round
    finalize_reserve_seconds=900,   # never start a round that leaves no time to commit it
    run_name="pfedes_20c_probe",
    cache="/kaggle/temp/veremi_cache",
    max_seconds=2.0 * 3600,   # 12 h hard cap; leave room to finalize artifacts
    require_resume=False,
)
# data_id is filled in below, once the feature order and scaler are known. It is part of
# proj/ckpt.py FINGERPRINT_KEYS, so a run cannot resume across a changed preprocessing.
for k, v in CFG.items(): print(f"{k:>20} = {v}")


In [ ]:
%%writefile /kaggle/working/proj/model.py
"""DAGSNet — Khan et al. 2025 §4.10, Eq. (38)-(48). 395.024 tham số.
Vào: (B, 66) đặc trưng đã z-score.  Ra: (B, 16) logit (CHƯA softmax).

pFedES uses the same class twice: `build_model` is a client's personalized classifier F_k
(16 logits) and `build_proxy` is the shared proxy feature extractor G (output dimension =
n_features, so x_hat = G(x) has the shape of x, as the paper's 'padding = same' extractor
requires). The project owner chose a DAGSNet-type proxy over the paper's 2-conv CNN.
"""
import torch
import torch.nn as nn


def cbr(i, o, k):
    """Conv → BatchNorm → ReLU. bias=False vì BatchNorm ngay sau đã có tham số dịch."""
    return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2, bias=False),
                         nn.BatchNorm1d(o), nn.ReLU(inplace=True))


class DenseNet1d(nn.Module):
    """Eq. (38)-(39): mỗi lớp nhận nối của toàn bộ feature map trước đó."""
    def __init__(self, cin, growth, layers):
        super().__init__()
        self.blocks = nn.ModuleList([cbr(cin + i * growth, growth, 3) for i in range(layers)])
        self.out_ch = cin + layers * growth

    def forward(self, x):
        for b in self.blocks:
            x = torch.cat([x, b(x)], dim=1)                     # Eq. (38)
        return x                                                # Eq. (39)


class Inception1d(nn.Module):
    """Eq. (40): bốn nhánh song song 1x1 / 3x3 / 5x5 / pool, nối lại."""
    def __init__(self, cin, c):
        super().__init__()
        self.b1 = cbr(cin, c, 1)
        self.b3 = nn.Sequential(cbr(cin, c, 1), cbr(c, c, 3))
        self.b5 = nn.Sequential(cbr(cin, c, 1), cbr(c, c, 5))
        self.bp = nn.Sequential(nn.MaxPool1d(3, 1, 1), cbr(cin, c, 1))
        self.out_ch = 4 * c

    def forward(self, x):
        return torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1)


class GoogleNet1d(nn.Module):
    """Eq. (40)-(41): các inception module xếp chồng."""
    def __init__(self, cin, modules_n, c=32):
        super().__init__()
        mods, ch = [], cin
        for _ in range(modules_n):
            m = Inception1d(ch, c); mods.append(m); ch = m.out_ch
        self.net = nn.Sequential(*mods); self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class AlexNet1d(nn.Module):
    """Eq. (42)-(44). ceil_mode=True: trục vị trí chỉ dài 11, không được để pool co về 0."""
    def __init__(self, cin, ch=128):
        super().__init__()
        self.net = nn.Sequential(
            cbr(cin, ch, 3), nn.MaxPool1d(2, ceil_mode=True),
            cbr(ch, ch, 3),  nn.MaxPool1d(2, ceil_mode=True),
            cbr(ch, ch, 3))
        self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class Fire1d(nn.Module):
    """Eq. (45)-(46): squeeze 1x1 nuôi hai nhánh expand 1x1 và 3x3."""
    def __init__(self, cin, sq, ex):
        super().__init__()
        self.squeeze = cbr(cin, sq, 1)                          # Eq. (46)
        self.e1 = cbr(sq, ex, 1)
        self.e3 = cbr(sq, ex, 3)
        self.out_ch = 2 * ex

    def forward(self, x):
        s = self.squeeze(x)
        return torch.cat([self.e1(s), self.e3(s)], dim=1)       # Eq. (45)


class SqueezeNet1d(nn.Module):
    def __init__(self, cin, modules_n, sq=32, ex=48):
        super().__init__()
        mods, ch = [], cin
        for _ in range(modules_n):
            m = Fire1d(ch, sq, ex); mods.append(m); ch = m.out_ch
        self.net = nn.Sequential(*mods); self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class DAGSNet(nn.Module):
    def __init__(self, cfg, n_features, out_dim):
        super().__init__()
        self.patch_len = cfg["patch_len"]
        assert n_features % self.patch_len == 0
        self.k = n_features // self.patch_len                   # 11
        cin = self.patch_len                                    # 6 kênh

        s = cfg["stem_ch"]
        self.stems   = nn.ModuleList([cbr(cin, s, 1) for _ in range(4)])
        self.dense   = DenseNet1d(s, cfg["dense_growth"], cfg["dense_layers"])
        self.google  = GoogleNet1d(s, cfg["incep_modules"])
        self.alex    = AlexNet1d(s)
        self.squeeze = SqueezeNet1d(s, cfg["fire_modules"])
        comb = self.dense.out_ch + self.google.out_ch + self.alex.out_ch + self.squeeze.out_ch

        self.head = nn.Sequential(                              # Eq. (48)
            nn.LayerNorm(comb), nn.Dropout(cfg["dropout"]),
            nn.Linear(comb, 256), nn.ReLU(inplace=True),
            nn.Dropout(cfg["dropout"]), nn.Linear(256, out_dim))

    def forward(self, x):                                       # (B, n_features)
        # view rồi MỚI transpose: gom 6 cột liên tiếp thành một patch, sau đó patch
        # mới trở thành trục vị trí. Làm view(B, 6, 11) thẳng sẽ trộn sai các cột.
        Fm = x.view(x.shape[0], self.k, self.patch_len).transpose(1, 2)   # (B, 6, 11)
        feats = [gp(stem(Fm)) for stem, gp in
                 zip(self.stems, [self.dense, self.google, self.alex, self.squeeze])]
        pooled = [f.mean(dim=-1) for f in feats]                # global average pool
        return self.head(torch.cat(pooled, dim=1))              # Eq. (47) -> (48)


# Cấu hình ĐÚNG như knowledge/architecture.md. Đổi bất kỳ giá trị nào ở đây thì
# state_dict sẽ không nạp được — đó là chủ ý.
CFG = {
    "patch_len": 6, "stem_ch": 96, "dense_growth": 32, "dense_layers": 3,
    "incep_modules": 2, "fire_modules": 3, "dropout": 0.1, "num_classes": 16,
}
N_PARAMS_MODEL = 395_024      # F_k: head 256 -> 16
N_PARAMS_PROXY = 407_874      # G:   head 256 -> 66, i.e. 395,024 - 4,112 + 16,962


def build_model(cfg):
    """Client classifier F_k. cfg is the dict saved in the checkpoint, so a model rebuilt
    here matches the one that produced those weights."""
    return DAGSNet({k: cfg[k] for k in CFG}, n_features=cfg["n_features"],
                   out_dim=cfg["num_classes"])


def build_proxy(cfg):
    """Proxy feature extractor G: same DAGSNet, output dimension n_features so that
    x_hat = G(x) has the shape of x (the paper's dimension condition)."""
    return DAGSNet({k: cfg[k] for k in CFG}, n_features=cfg["n_features"],
                   out_dim=cfg["n_features"])


In [ ]:
%%writefile /kaggle/working/proj/ckpt.py
"""Weights, resume state and the completion marker — three files, one atomic round.

The per-round weights file holds WEIGHTS ONLY and loads with weights_only=True: the global
proxy extractor G and every client's personalized F_k as plain state_dict tensors. RNG and
the round counter live in a separate resume bundle keyed by the same round, so a reader
never has to unpickle arbitrary objects to look at a checkpoint.

pFedES persistent state is the round, theta (G) and the N client models. AdamW is
re-created per client per round (owner's decision, see pfedes.py), so no optimizer state
exists at a round boundary; client selection is a pure function of (seed, round).
"""
import csv, hashlib, json, os, random, shutil
from pathlib import Path
import numpy as np, torch

SUBDIRS = ("weights", "resume", "complete", "metrics", "preds", "confusion", "reports", "logs")

# What a later session imports. `preds` and `logs` are not needed to CONTINUE, but a run
# split across sessions must still be verifiable from its final output alone.
RESUME_SUBDIRS = ("weights", "resume", "metrics", "confusion", "preds", "logs")

# Every input that changes what the numbers mean. `rounds` IS in the list since the
# per-round LR schedule (proj.pfedes.lr_at) spans the whole run: the weights at round r
# depend on how many rounds were planned, so a continuation push must plan the same
# total. Paths, world_size, compile, eval_batch, max_seconds and require_resume are
# operational and absent.
FINGERPRINT_KEYS = (
    "patch_len", "stem_ch", "dense_growth", "dense_layers", "incep_modules",
    "fire_modules", "dropout", "num_classes", "n_features",
    "lr", "lr_schedule", "lr_min", "rounds", "weight_decay", "mu", "clip",
    "n_clients", "participation", "batch", "local_epochs", "proxy_epochs", "seed",
    "data_id", "run_name",
)


def run_dir(run_name, base="/kaggle/working/runs"):
    d = Path(base) / run_name
    for s in SUBDIRS: (d / s).mkdir(parents=True, exist_ok=True)
    return d


def fingerprint(cfg):
    """A missing key is a KeyError, never a default. Silently hashing `None` for a key that
    was renamed is exactly how a fingerprint stops protecting anything."""
    missing = [k for k in FINGERPRINT_KEYS if k not in cfg]
    if missing:
        raise KeyError(f"fingerprint needs {missing} in CFG; add them, do not default them")
    payload = json.dumps({k: cfg[k] for k in FINGERPRINT_KEYS}, sort_keys=True, default=str)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def file_sha(path):
    """Hash of the file as written. Not a re-serialisation: torch.save embeds a zip whose
    bytes are not reproducible, so only the bytes on disk are a stable identity."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def atomic_save(obj, path):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp); os.replace(tmp, path)     # replace is atomic on POSIX


def atomic_np_save(path, arr):
    """np.save appends .npy to a name that lacks it, so write through a handle."""
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "wb") as f:
        np.save(f, arr); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)


# --- RNG kept as tensors and primitives so the resume bundle also loads weights_only=True.
def rng_state():
    npy = np.random.get_state()
    return {"python": random.getstate(),
            "numpy": (npy[0], torch.from_numpy(npy[1].copy()), int(npy[2]), int(npy[3]),
                      float(npy[4])),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def set_rng_state(s):
    random.setstate(tuple(s["python"]))
    n = s["numpy"]
    np.random.set_state((n[0], n[1].numpy().astype(np.uint32), n[2], n[3], n[4]))
    torch.set_rng_state(s["torch"].cpu())
    if s.get("cuda") is not None: torch.cuda.set_rng_state_all(s["cuda"])


def cpu_sd(model):
    """A detached CPU copy of a module's state_dict: tensors only, so the file it goes
    into loads with weights_only=True."""
    core = getattr(model, "_orig_mod", model)                 # unwrap torch.compile
    return {k: v.detach().cpu().clone() for k, v in core.state_dict().items()}


# --------------------------------------------------------------------------- write
def save_round_weights(global_sd, clients_sd, rnd, cfg, metrics, d):
    """global_sd: state_dict of G at theta^t; clients_sd: {cid: state_dict of F_k at
    w_k^t} for ALL N clients (the ones not selected this round carry their previous
    weights). The caller writes the marker, and only after every other artifact is on
    disk."""
    prev = d / "weights" / f"round_{rnd - 1:03d}.pt"
    # The hash of the weights this round was trained FROM. Two runs of the same config have
    # the same fingerprint, so without this an import can keep round 1 of run A and take
    # round 2 of run B and call the result one training history.
    prev_sha = file_sha(prev) if rnd > 1 and prev.is_file() else None
    atomic_save({"round": int(rnd),
                 "global": global_sd,
                 "clients": {int(c): sd for c, sd in sorted(clients_sd.items())},
                 "cfg": {k: v for k, v in cfg.items()},       # rebuild recipe
                 "fingerprint": fingerprint(cfg),
                 "prev_sha": prev_sha,
                 "metrics": metrics},
                d / "weights" / f"round_{rnd:03d}.pt")
    # The RNG here is the DRIVER's, and the driver does not train. Recorded for forensics:
    # worker training RNG is re-derived from (seed, round, client), client selection from
    # (seed, round), so continuation does not depend on restoring this.
    atomic_save({"round": int(rnd), "rng": rng_state(),
                 "note": "no optimizer state: AdamW is re-created per client per round; "
                         "worker RNG derives from (seed, round, client)",
                 "fingerprint": fingerprint(cfg)},
                d / "resume" / f"round_{rnd:03d}.pt")
    return d / "weights" / f"round_{rnd:03d}.pt"


def mark_complete(d, rnd):
    (d / "complete" / f"round_{rnd:03d}.done").write_text("")


# --------------------------------------------------------------------------- rebuild
def _load_into(model, sd, expect_params):
    sd = {k.removeprefix("module.").removeprefix("_orig_mod."): v for k, v in sd.items()}
    bad = [k for k, v in sd.items() if v.is_floating_point() and not torch.isfinite(v).all()]
    if bad:
        raise RuntimeError(f"non-finite values in checkpoint tensors {bad[:3]}")
    model.load_state_dict(sd, strict=True)
    n = sum(p.numel() for p in model.parameters())
    if expect_params is not None and n != expect_params:
        raise RuntimeError(f"rebuilt {n:,} parameters, expected {expect_params:,}")
    return model.eval()


def load_weights(path, build_model, build_proxy, expect_model=None, expect_proxy=None,
                 clients=None, device="cpu"):
    """Rebuild G and the client models at exactly this checkpoint, from weights alone.

    Every assert here exists because its failure is otherwise silent: strict=True catches a
    filtered running_mean/var (eval() would then normalize by 0/1 and report nothing), the
    parameter count catches a cfg that drifted from the one that trained these weights, and
    the finite check catches a diverged tensor that argmax would turn into a plausible label.
    `clients=None` rebuilds every client; pass a list to rebuild a subset."""
    ck = torch.load(path, map_location="cpu", weights_only=True)
    if ck.get("fingerprint") != fingerprint(ck["cfg"]):
        raise RuntimeError("weights file fingerprint disagrees with its own cfg")
    G = _load_into(build_proxy(ck["cfg"]), ck["global"], expect_proxy).to(device)
    want = sorted(ck["clients"]) if clients is None else list(clients)
    if sorted(ck["clients"]) != list(range(ck["cfg"]["n_clients"])):
        raise RuntimeError(f"checkpoint holds clients {sorted(ck['clients'])[:5]}..., "
                           f"expected 0..{ck['cfg']['n_clients'] - 1}")
    Fs = {int(c): _load_into(build_model(ck["cfg"]), ck["clients"][c], expect_model).to(device)
          for c in want}
    return G, Fs, ck


# --------------------------------------------------------------------------- verify
def round_ok(d, rnd, fp=None, chain=True):
    """A round counts only if every artifact of that round is present, READABLE, and links
    to the round before it. The marker alone proves nothing."""
    w = d / "weights" / f"round_{rnd:03d}.pt"
    r = d / "resume" / f"round_{rnd:03d}.pt"
    m = d / "metrics" / f"round_{rnd:03d}.json"
    c = d / "confusion" / f"round_{rnd:03d}.npy"
    for path in (w, r, m, c):
        if not path.is_file() or path.stat().st_size == 0:
            return False
    try:
        ck = torch.load(w, map_location="cpu", weights_only=True, mmap=True)
        rs = torch.load(r, map_location="cpu", weights_only=True)   # not just "it exists"
        row = json.loads(m.read_text())
        np.load(c)
    except Exception:
        return False                      # truncated or corrupt reads as a failed round
    if int(ck.get("round", -1)) != rnd or int(row.get("round", -1)) != rnd:
        return False
    if int(rs.get("round", -1)) != rnd:
        return False
    if fp is not None and (ck.get("fingerprint") != fp or rs.get("fingerprint") != fp):
        return False
    if chain:
        # Round r is only meaningful as the product of round r-1. Same config, same
        # fingerprint, different training history -> different bytes -> chain breaks here.
        prev = d / "weights" / f"round_{rnd - 1:03d}.pt"
        want = file_sha(prev) if rnd > 1 and prev.is_file() else None
        if ck.get("prev_sha") != want:
            return False
    return True


def _markers(d):
    return sorted(int(p.stem.split("_")[1]) for p in (d / "complete").glob("round_*.done"))


def last_complete_round(d, fp=None):
    """Largest r such that rounds 1..r are ALL complete. A gap ends the run."""
    last = 0
    top = _markers(d)[-1] if _markers(d) else 0
    for r in range(1, top + 1):
        if not (d / "complete" / f"round_{r:03d}.done").is_file() or not round_ok(d, r, fp):
            break
        last = r
    return last or None


# --------------------------------------------------------------------------- resume
def _attached_source(run_name, attached=Path("/kaggle/input")):
    if not attached.exists():
        return None
    roots = sorted({p.parent for p in attached.rglob("complete/round_*.done")
                    if run_name in p.parts})
    if len(roots) > 1:
        raise RuntimeError(f"Multiple resume trees for {run_name}: {roots}")
    return roots[0].parent if roots else None


def _import_from(src, d, fp):
    """Copy through a staging tree, verify there, then publish one round at a time with its
    marker last. Idempotent: a crash mid-publish leaves that round unmarked, and the next
    attempt re-copies it from the still-mounted source. The source's own markers bound the
    import, and the destination must not already hold a different history."""
    stage = d.parent / f".{d.name}.import"
    shutil.rmtree(stage, ignore_errors=True)
    for sub in RESUME_SUBDIRS + ("complete",):
        (stage / sub).mkdir(parents=True, exist_ok=True)
    for sub in RESUME_SUBDIRS + ("complete",):
        peer = src / sub
        if not peer.is_dir(): continue
        for f in peer.iterdir():
            if f.is_file():
                # copyfile, not copy2: a read-only mount's mode would carry across and the
                # first rewrite would die with PermissionError.
                shutil.copyfile(f, stage / sub / f.name)
                os.chmod(stage / sub / f.name, 0o644)

    src_last = 0
    while ((stage / "complete" / f"round_{src_last + 1:03d}.done").is_file()
           and round_ok(stage, src_last + 1, fp)):
        src_last += 1

    for r in range(1, src_last + 1):
        here = d / "weights" / f"round_{r:03d}.pt"
        if here.is_file() and file_sha(here) != file_sha(stage / "weights" / f"round_{r:03d}.pt"):
            shutil.rmtree(stage, ignore_errors=True)
            raise RuntimeError(
                f"round {r} in {d} and in {src} have the same config but different weights: "
                "these are two different training runs, not one interrupted one. Refusing to "
                "splice them. Detach one source, or start a new run_name.")

    published = 0
    for r in range(1, src_last + 1):
        if not round_ok(d, r, fp):
            for sub in RESUME_SUBDIRS:
                for f in (stage / sub).glob(f"round_{r:03d}.*"):
                    shutil.copyfile(f, d / sub / f.name)
                    os.chmod(d / sub / f.name, 0o644)
        if not (d / "complete" / f"round_{r:03d}.done").is_file():
            mark_complete(d, r)                     # marker last, per round
        published = r
    shutil.rmtree(stage, ignore_errors=True)
    n_mark = len(list((src / "complete").glob("round_*.done"))) if (src / "complete").is_dir() else 0
    print(f"[resume] {src}: {n_mark} marker(s), {src_last} verified, imported 1..{published}"
          if published else
          f"[resume] {src} held no verifiable committed round; starting from 0")
    return published


def resolve_resume(run_name, cfg=None, attached=Path("/kaggle/input"), d=None):
    """working/ first, then any attached input (previous kernel output or a checkpoint
    dataset). Returns the last round that is complete AND verified, or None."""
    d = run_dir(run_name) if d is None else d
    fp = fingerprint(cfg) if cfg is not None else None
    src = _attached_source(run_name, attached)
    if src is not None:
        _import_from(src, d, fp)
    rebuild_history(d, fp)
    return last_complete_round(d, fp)


# --------------------------------------------------------------------------- history
def _write_csv(p, rows):
    tmp = Path(p).with_suffix(".csv.tmp")
    with open(tmp, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader(); w.writerows(rows)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, p)                      # a crash mid-write cannot truncate the live file


def rebuild_history(d, fp=None):
    """history.csv (mean over clients per round) and clients.csv (one row per client per
    round) are DERIVED from metrics/round_NNN.json, never authoritative. Only the verified
    contiguous range 1..last goes in."""
    last = last_complete_round(d, fp) or 0
    rows, crow = [], []
    for r in range(1, last + 1):
        p = d / "metrics" / f"round_{r:03d}.json"
        if not p.is_file(): break
        try: j = json.loads(p.read_text())
        except Exception: break
        rows.append({k: v for k, v in j.items() if k not in ("clients", "per_class")})
        crow += [{"round": r, **{k: v for k, v in c.items() if k != "per_class"}}
                 for c in j["clients"]]
    if rows:
        _write_csv(d / "history.csv", rows)
        _write_csv(d / "clients.csv", crow)
    else:
        for n in ("history.csv", "clients.csv"):
            if (d / n).exists(): (d / n).unlink()  # a stale CSV outlives the rounds it described
    return len(rows)


def append_history(d, row, client_rows):
    """Keyed by round: a redone round replaces its lines instead of duplicating them."""
    p = d / "history.csv"
    rows = {}
    if p.exists():
        with open(p) as f:
            rows = {int(r["round"]): r for r in csv.DictReader(f)}
    rows[int(row["round"])] = {k: str(v) for k, v in row.items()}
    _write_csv(p, [rows[k] for k in sorted(rows)])
    q = d / "clients.csv"
    old = []
    if q.exists():
        with open(q) as f:
            old = [r for r in csv.DictReader(f) if int(r["round"]) != int(row["round"])]
    _write_csv(q, old + [{k: str(v) for k, v in c.items()} for c in client_rows])


In [ ]:
%%writefile /kaggle/working/proj/metrics.py
"""All 10 metrics from a full confusion matrix. No batch averaging, no sampling."""
import numpy as np

METRIC_KEYS = ("accuracy", "precision_macro", "precision_micro", "precision_weighted",
               "recall_macro", "recall_micro", "recall_weighted",
               "f1_macro", "f1_micro", "f1_weighted")


def metrics_from_confusion(cm):
    """cm[i, j] = count of true class i predicted as j. Integer counts in, 10 floats out."""
    cm = np.asarray(cm, dtype=np.float64)
    tp = np.diag(cm)
    support = cm.sum(axis=1)                       # true count per class
    pred = cm.sum(axis=0)                          # predicted count per class
    total = cm.sum()

    # A class never predicted has precision 0/0; sklearn defines it as 0 with zero_division=0.
    prec = np.divide(tp, pred, out=np.zeros_like(tp), where=pred > 0)
    rec = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    denom = prec + rec
    f1 = np.divide(2 * prec * rec, denom, out=np.zeros_like(tp), where=denom > 0)

    acc = tp.sum() / total
    w = support / total                            # weighted = support-weighted mean
    # float(), not np.float64: a numpy scalar anywhere in a checkpoint dict makes
    # torch.load(weights_only=True) refuse the whole file, and the failure only appears
    # when something later tries to read it back.
    return {k: float(v) for k, v in (
        ("accuracy", acc),
        ("precision_macro", prec.mean()), ("precision_micro", acc),
        ("precision_weighted", (prec * w).sum()),
        ("recall_macro", rec.mean()), ("recall_micro", acc),
        ("recall_weighted", (rec * w).sum()),
        ("f1_macro", f1.mean()), ("f1_micro", acc),
        ("f1_weighted", (f1 * w).sum()))}


def per_class_from_confusion(cm, class_names):
    cm = np.asarray(cm, dtype=np.float64)
    tp, support, pred = np.diag(cm), cm.sum(axis=1), cm.sum(axis=0)
    prec = np.divide(tp, pred, out=np.zeros_like(tp), where=pred > 0)
    rec = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    d = prec + rec
    f1 = np.divide(2 * prec * rec, d, out=np.zeros_like(tp), where=d > 0)
    return [{"idx": i, "class": class_names[i], "support": int(support[i]),
             "precision": float(prec[i]), "recall": float(rec[i]), "f1": float(f1[i])}
            for i in range(len(class_names))]


In [ ]:
%%writefile /kaggle/working/proj/data.py
"""Parquet -> resident fp16 tensors. One pass per session, then no input pipeline at all.

Two facts from knowledge/dataset.md that produce silent corruption if ignored:
  * train/ is ALREADY z-scored; test/ is NOT. Applying scaler.json to train a second time
    destroys it and raises nothing.
  * the integer label is in column `label` (int8, 0..15). Decoding `attack_type` strings
    over 43 M rows costs tens of seconds per pass for the same information.
"""
import json
from pathlib import Path
import numpy as np
import pyarrow.dataset as ds

CACHE_FILES = ("train_X.f16.npy", "train_y.u8.npy", "test_X.f16.npy",
               "test_y.u8.npy", "spans.json")


def find_root(sentinel, bases=("/kaggle/input",)):
    """Kaggle mounts are nested by kind and owner; the prefix is not /kaggle/input/<slug>/.
    Resolve by locating the sentinel instead of hard-coding a depth."""
    depth = len(Path(sentinel).parts)          # strip the whole sentinel, not one level
    hits = []
    for b in bases:
        p = Path(b)
        if p.exists():
            for q in p.rglob(sentinel):
                if q.is_dir():
                    r = q
                    for _ in range(depth): r = r.parent
                    hits.append(r)
    hits = sorted(set(hits))
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {sentinel!r} under {bases}, got {hits}")
    return hits[0]


def parquet_files(root):
    """The parquet parts of a directory, in a fixed order, and NOTHING else.

    `ds.dataset(dir, format="parquet")` opens every file it finds. The centralized test
    directory ships a `part-NNNNN.stats.json` sidecar next to each part, so `to_table()`
    died on Kaggle with "Parquet magic bytes not found in footer" after a nine-minute train
    decode. It survived locally only because the smoke fixture reads through `to_batches()`
    and breaks early, never reaching a sidecar -- a lazy reader hides exactly this.

    sorted() is not cosmetic either: the fragment order fixes the row order of the test set,
    and therefore the order of y_true and of every saved prediction vector."""
    files = sorted(str(f) for f in Path(root).rglob("*.parquet"))
    if not files:
        raise RuntimeError(f"no .parquet files under {root}")
    return files


def _labels(col, num_classes=16):
    """astype(np.uint8) on a label of -1 gives 255 and on 300 gives 44 -- both are silent,
    and both survive every downstream assert because the counts still add up."""
    v = col.to_numpy(zero_copy_only=False)
    lo, hi = int(v.min()), int(v.max())
    if lo < 0 or hi >= num_classes:
        raise RuntimeError(f"labels out of range [{lo}, {hi}], expected 0..{num_classes-1}")
    return v.astype(np.uint8)


def load_clients(fl_root, feature_cols, n_clients, dtype=np.float16):
    """Returns X (N,66) fp16, y (N,) uint8, and spans[cid] = (lo, hi) contiguous row range.

    Contiguous spans are what make the training loop a slice + randperm instead of a
    gather over a client-id column."""
    dirs = sorted((fl_root / "train").glob("client_id=*"),
                  key=lambda p: int(p.name.split("=")[1]))
    if len(dirs) != n_clients:
        raise RuntimeError(f"expected {n_clients} client dirs, found {len(dirs)}")
    cols = list(feature_cols) + ["label"]
    xs, ys, spans, off = [], [], {}, 0
    for d in dirs:
        cid = int(d.name.split("=")[1])
        t = ds.dataset(parquet_files(d), format="parquet").to_table(columns=cols)
        n = t.num_rows
        a = np.empty((n, len(feature_cols)), dtype=dtype)
        for j, c in enumerate(feature_cols):
            a[:, j] = t.column(c).to_numpy(zero_copy_only=False).astype(dtype, copy=False)
        xs.append(a)
        ys.append(_labels(t.column("label")))
        spans[cid] = (off, off + n); off += n
        del t
    return np.concatenate(xs), np.concatenate(ys), spans


def load_test(test_root, feature_cols, scaler, dtype=np.float16):
    """test/ is raw: apply scaler.json here, and nowhere else."""
    t = ds.dataset(parquet_files(test_root), format="parquet").to_table(
        columns=list(feature_cols) + ["label"])
    n = t.num_rows
    X = np.empty((n, len(feature_cols)), dtype=dtype)
    for j, c in enumerate(feature_cols):
        v = t.column(c).to_numpy(zero_copy_only=False).astype(np.float64)
        v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
        s = scaler[c]
        X[:, j] = ((v - s["mean"]) / s["std_used"]).astype(dtype)
    y = _labels(t.column("label"))
    return X, y


def assert_fp16_safe(X, name):
    """knowledge/dataset.md measured max|x| = 570.44 on both splits, two orders below the
    fp16 ceiling. Assert it rather than inherit the assumption."""
    m = float(np.abs(X).max())
    if not np.isfinite(m) or m >= 65504:
        raise RuntimeError(f"{name}: max|x| = {m} is not fp16-safe")
    return m


def cache_ok(cache, want, n_clients, n_features=66):
    """Is the prepack cache complete, current and self-consistent?

    A matching manifest is a claim, not evidence. Running the notebook's own hit branch with
    a matching manifest and no train_X printed "cache reusable"; the worker then died opening
    a file that had never been written. Headers are read through mmap, so this costs a few
    stat calls and no data."""
    cache = Path(cache)
    mf = cache / "manifest.json"
    if not mf.is_file():
        return False
    try:
        if json.loads(mf.read_text()) != want:
            return False
        for f in CACHE_FILES:
            if not (cache / f).is_file() or (cache / f).stat().st_size == 0:
                return False
        spans = {int(k): tuple(v)
                 for k, v in json.load(open(cache / "spans.json")).items()}
        X = np.load(cache / "train_X.f16.npy", mmap_mode="r")
        Y = np.load(cache / "train_y.u8.npy", mmap_mode="r")
        TX = np.load(cache / "test_X.f16.npy", mmap_mode="r")
        TY = np.load(cache / "test_y.u8.npy", mmap_mode="r")
    except Exception as e:
        print("[cache] unreadable:", e)
        return False
    n = sum(hi - lo for lo, hi in spans.values())
    r = sorted(spans.values())
    return bool(
        len(spans) == n_clients
        and X.dtype == np.float16 and Y.dtype == np.uint8
        and TX.dtype == np.float16 and TY.dtype == np.uint8
        and X.shape == (n, n_features) and Y.shape == (n,)
        and TX.ndim == 2 and TX.shape[1] == n_features and len(TX) == len(TY)
        and r and r[0][0] == 0 and r[-1][1] == n
        and all(a[1] == b[0] for a, b in zip(r, r[1:])))


In [ ]:
%%writefile /kaggle/working/proj/pfedes.py
"""pFedES client update and aggregation — Yi et al., Eq. (4)-(11).

Round t, client k in S^t (K = C*N clients drawn without replacement, seeded by round):
  Step 1  freeze G(theta^{t-1}), train F_k:
            x_hat = G(x)                                              Eq. (4)
            l_w   = mu * CE(F_k(x_hat), y) + (1 - mu) * CE(F_k(x), y)  Eq. (5)-(6)
            w_k^t <- w_k^{t-1} - eta_w * grad l_w                      Eq. (7)
  Step 2  freeze F_k(w_k^t), train G:
            l_theta = CE(F_k(G(x)), y)                                 Eq. (8)-(9)
            theta_k^t <- theta^{t-1} - eta_theta * grad l_theta        Eq. (10)
Server:   theta^t = sum_{k in S^t} (n_k / sum_{j in S^t} n_j) * theta_k^t   Eq. (11)

Implementation choices that are NOT in the paper and must be reported:
  * The two forward passes of Step 1 run as ONE forward on the concatenated batch
    [x_hat; x] (2B rows). The loss is exactly Eq. (6) -- each half gets its own CE mean --
    but F_k's BatchNorm sees the union of enhanced and original rows in one batch. Two
    separate calls of a CUDA-graph-compiled module before backward are not allowed (the
    second replay overwrites the first's outputs), and the launch-bound DAGSNet makes the
    concatenated forward ~1.7x cheaper than two calls.
  * "Frozen" is implemented as eval() + no parameter gradients: the frozen module uses its
    running BatchNorm statistics and no dropout, so x_hat in Step 1 and F_k in Step 2 are
    deterministic functions of the input.
  * Eq. (11) normalises over the SELECTED clients. The paper writes n_k / n with n the total
    over all N clients (Preliminaries); with C < 100 % that literal form shrinks theta by
    the factor C every round.
  * Optimizer is AdamW (owner's decision, not the paper's SGD), re-created per client per
    round: theta arrives fresh from the server each round and the owner chose to reset the
    F_k moments as well, so no optimizer state is ever persisted.
  * The learning rate follows a per-ROUND schedule (`lr_at`), constant within a round and
    shared by eta_w and eta_theta. The paper uses a constant 0.01; the owner's unified
    choice across the sibling rebuilds (2026-09-13) is a cosine from cfg['lr'] at round 1
    to cfg['lr_min'] at the last round, because every personalized F_k on this data peaks
    on the global test after ~2 local epochs and then drifts under a constant rate.
"""
import contextlib
import math
import numpy as np
import torch
import torch.nn.functional as F


def amp(cfg):
    """fp16 autocast on CUDA; a no-op on CPU so the same code runs in the local
    simulation. Never bf16: the T4 is sm_75 and falls back to a slow emulation path."""
    if cfg.get("device", "cuda") == "cuda":
        return torch.autocast("cuda", dtype=torch.float16)
    return contextlib.nullcontext()


# --------------------------------------------------------------------- flat layout
# One flat float vector + one int vector per model. A DAGSNet state_dict has 192 entries;
# torch.multiprocessing gives each tensor its own shared-memory fd, so 100 clients a round
# would exhaust the process fd limit. Parameters come FIRST so vec[:n_params] is exactly
# the learnable block; BN running stats follow as buffers.
def layout(model):
    pnames = {n for n, _ in model.named_parameters()}
    sd = model.state_dict()
    fkeys = [k for k in sd if k in pnames]
    fkeys += [k for k in sd if k not in pnames and sd[k].is_floating_point()]
    ikeys = [k for k in sd if not sd[k].is_floating_point()]
    n_params = sum(sd[k].numel() for k in fkeys if k in pnames)
    return fkeys, ikeys, n_params


def flatten(model, fkeys, ikeys):
    sd = model.state_dict()
    fv = torch.cat([sd[k].reshape(-1).float() for k in fkeys])
    iv = torch.stack([sd[k].reshape(-1).long().squeeze() for k in ikeys]) if ikeys \
        else torch.zeros(0, dtype=torch.long)
    return fv, iv


def unflatten_into(model, fv, iv, fkeys, ikeys):
    sd = model.state_dict()
    o = 0
    for k in fkeys:
        t = sd[k]; n = t.numel()
        t.copy_(fv[o:o + n].view_as(t)); o += n           # copy_ keeps addresses -> CUDA graph valid
    for j, k in enumerate(ikeys):
        sd[k].copy_(iv[j].view_as(sd[k]))
    return model


# --------------------------------------------------------------------- client sampling
def n_selected(n_clients, frac):
    """K = |C * N|, at least 1. frac = 1.0 selects everyone."""
    k = int(round(frac * n_clients))
    if not 1 <= k <= n_clients:
        raise ValueError(f"participation {frac} of {n_clients} clients gives K={k}")
    return k


def select_clients(n_clients, frac, seed, rnd):
    """The paper's 'randomly selects K clients among N'. A pure function of (seed, round):
    a resumed session draws the same set the original would have, and the draw does not
    consume the training RNG."""
    k = n_selected(n_clients, frac)
    if k == n_clients:
        return list(range(n_clients))
    g = np.random.default_rng(seed * 1_000_003 + rnd * 10_007)
    return sorted(int(c) for c in g.choice(n_clients, size=k, replace=False))


# --------------------------------------------------------------------- learning rate
LR_SCHEDULES = ("constant", "cosine")


def lr_at(cfg, rnd):
    """Learning rate of round `rnd` (1-based), held constant within the round and used for
    both eta_w and eta_theta. A pure function of (cfg, rnd): a resumed session applies
    exactly the value the original session would have.

      constant : cfg['lr'] every round
      cosine   : lr_min + (lr - lr_min)/2 * (1 + cos(pi * (rnd-1) / (rounds-1)))
                 -- cfg['lr'] at round 1, cfg['lr_min'] at round cfg['rounds']; the
                 formula the afpha rebuild uses, so the sibling projects share one schedule.
    """
    sched = cfg.get("lr_schedule", "constant")
    if sched == "constant":
        return float(cfg["lr"])
    if sched == "cosine":
        T = int(cfg["rounds"])
        if not 1 <= rnd <= T:
            raise ValueError(f"round {rnd} outside 1..{T}: the cosine schedule is undefined")
        if T == 1:
            return float(cfg["lr"])
        lo, hi = float(cfg["lr_min"]), float(cfg["lr"])
        return lo + 0.5 * (hi - lo) * (1.0 + math.cos(math.pi * (rnd - 1) / (T - 1)))
    raise ValueError(f"lr_schedule {sched!r} not in {LR_SCHEDULES}")


# --------------------------------------------------------------------- client update
def _freeze(module, frozen):
    """frozen: eval() so BatchNorm uses running stats and Dropout is off, and no gradient
    is accumulated into its parameters (the backward still flows THROUGH it to its input,
    which is what Step 2 needs). Trainable: the reverse."""
    module.train(not frozen)
    for p in module.parameters():
        p.requires_grad_(not frozen)


def _epoch(Fc, Fe, Gc, Ge, opt, scaler, params, X, Y, lo, hi, cfg, gen, step_fn, accs):
    """One pass over the client's rows. `step_fn(Fm, Gm, xb, yb)` returns (loss, extra)
    for the phase being trained; `accs` are device-side accumulators updated per step.
    The tail batch is a different shape and would recompile the CUDA graph once per
    client, so it runs on the eager modules: same weights, same math."""
    B, clip, dev = cfg["batch"], cfg["clip"], X.device
    zero = torch.zeros((), device=dev)
    perm = lo + torch.randperm(hi - lo, generator=gen, device=dev)
    n = 0
    for i in range(0, hi - lo, B):
        # One training step invokes up to four captured graphs (G forward, F forward and
        # backward, and in Step 2 G's backward). CUDA-graph trees decide which pool memory
        # is dead from the iteration boundary, so declare it explicitly instead of letting
        # the no_grad G forward be mistaken for a new iteration mid-step.
        if X.is_cuda:
            torch.compiler.cudagraph_mark_step_begin()
        idx = perm[i:i + B]
        full = idx.numel() == B
        xb = X[idx].float()
        yb = Y[idx].long()
        loss, extra = step_fn(Fc if full else Fe, Gc if full else Ge, xb, yb)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)                                   # grads now in true units
        gn = torch.nn.utils.clip_grad_norm_(params, clip)
        prev = scaler._scale.clone() if scaler.is_enabled() else None
        scaler.step(opt); scaler.update()
        opt.zero_grad(set_to_none=True)
        # A skipped step overflowed: its grad-norm is inf and its loss may be nan.
        # torch.where, not multiplication -- inf*0 is nan.
        applied = (scaler._scale >= prev) if prev is not None \
            else torch.ones((), dtype=torch.bool, device=dev)
        accs["nonfin"] += (~torch.isfinite(gn) & applied).float()
        accs["loss"] += torch.where(applied, loss.detach(), zero)
        accs["extra"] += torch.where(applied, extra.detach(), zero)
        accs["gn"] += torch.where(applied, gn, zero)
        accs["skips"] += (~applied).float()
        n += 1
    return n


def client_update(Fc, Fe, Gc, Ge, optF, optG, scF, scG, X, Y, lo, hi, cfg, gen):
    """Step 1 then Step 2 for one client. Fe/Ge are the eager modules that own the
    parameters; Fc/Gc are their compiled aliases (or the same objects when not compiled).

    Returns two dicts of device-side accumulators (read once by the caller) and the step
    counts. Dropout reads torch's default generator, which the worker re-seeds from
    (seed, round, client) before calling this; `gen` drives only the shuffles."""
    mu, dev = cfg["mu"], X.device
    B = cfg["batch"]
    paramsF = list(Fe.parameters())
    paramsG = list(Ge.parameters())

    def new_accs():
        return {k: torch.zeros((), device=dev) for k in ("loss", "extra", "gn", "skips",
                                                         "nonfin")}

    # ---- Step 1: freeze G, train F_k on [x_hat; x]                       Eq. (4)-(7)
    _freeze(Ge, True); _freeze(Fe, False)

    def step1(Fm, Gm, xb, yb):
        with torch.no_grad(), amp(cfg):
            xhat = Gm(xb)
        with amp(cfg):
            z = Fm(torch.cat([xhat.float(), xb], dim=0))
        z = z.float()
        n = xb.shape[0]
        l1 = F.cross_entropy(z[:n], yb)                      # enhanced data, Eq. (5)
        l2 = F.cross_entropy(z[n:], yb)                      # original data, Eq. (5)
        return mu * l1 + (1.0 - mu) * l2, l2                 # Eq. (6); l2 reported separately

    acc1 = new_accs()
    n1 = sum(_epoch(Fc, Fe, Gc, Ge, optF, scF, paramsF, X, Y, lo, hi, cfg, gen, step1, acc1)
             for _ in range(cfg["local_epochs"]))

    # ---- Step 2: freeze F_k(w_k^t), train G                             Eq. (8)-(10)
    _freeze(Fe, True); _freeze(Ge, False)

    def step2(Fm, Gm, xb, yb):
        with amp(cfg):
            z = Fm(Gm(xb).float())
        loss = F.cross_entropy(z.float(), yb)                # Eq. (9)
        return loss, loss

    acc2 = new_accs()
    n2 = sum(_epoch(Fc, Fe, Gc, Ge, optG, scG, paramsG, X, Y, lo, hi, cfg, gen, step2, acc2)
             for _ in range(cfg["proxy_epochs"]))

    _freeze(Fe, False); _freeze(Ge, False)                  # leave both trainable
    return acc1, n1, acc2, n2


def expected_steps(n_k, cfg):
    per_epoch = math.ceil(n_k / cfg["batch"])
    return cfg["local_epochs"] * per_epoch, cfg["proxy_epochs"] * per_epoch


# --------------------------------------------------------------------- aggregation
def aggregate(updates):
    """Eq. (11) over the selected set: theta = sum_k (n_k / n_S) theta_k. `updates` must
    already be sorted by client id -- float addition order decides the result, and it must
    not be set by a completion race."""
    n_s = sum(n_k for _, n_k, _, _ in updates)
    acc = None
    for cid, n_k, fv, iv in updates:
        w = n_k / n_s
        acc = fv * w if acc is None else acc.add_(fv, alpha=w)
    # num_batches_tracked is an int counter, not an averageable quantity; with the default
    # BatchNorm momentum=0.1 it is unused at inference. Take the max so it stays monotone.
    ints = torch.stack([iv for _, _, _, iv in updates]).amax(dim=0) if updates[0][3].numel() \
        else updates[0][3]
    return acc, ints


In [ ]:
%%writefile /kaggle/working/proj/evaluate.py
"""Per-client evaluation of F_k(x) on the fixed global test set.

pFedES infers with the personalized local model alone ("only each client's personalized
heterogeneous local model F_k(w_k) is used for inference"), so a round's evaluation is N
independent full-test passes -- at 100 clients the dominant cost of the run.

Two exact speed-ups, both measured on 2xT4 in this repository's sibling projects:
  * BatchNorm folding: in eval mode BN is an affine map with constant coefficients, so it
    folds into the preceding Conv1d exactly (max|dlogit| 4.8e-07 measured), removing 31
    kernel launches per forward.
  * One folded TEMPLATE module per worker, compiled once with CUDA graphs. Client weights
    are folded and copied INTO the template with load_state_dict (in-place copy_), so
    parameter addresses never change and the captured graph stays valid for every client.
"""
import copy
import torch
import torch.nn as nn


@torch.no_grad()
def fold_bn(model):
    """Return an eval-mode copy with every BatchNorm folded into its preceding Conv1d.

        y = gamma*(conv(x) - mean)/sqrt(var + eps) + beta
          = conv'(x) + b'   with   w' = w*gamma/sqrt(var+eps),  b' = beta - gamma*mean/sqrt(var+eps)

    Exact for model.eval(); meaningless for model.train(). Every BatchNorm in DAGSNet sits
    inside a `cbr` block, i.e. Sequential(Conv1d, BN, ReLU), and the assertion at the end
    is what stops a future architecture change from silently leaving one unfolded."""
    m = copy.deepcopy(model).eval()
    for seq in m.modules():
        if not (isinstance(seq, nn.Sequential) and len(seq) >= 2
                and isinstance(seq[0], nn.Conv1d) and isinstance(seq[1], nn.BatchNorm1d)):
            continue
        conv, bn = seq[0], seq[1]
        inv = torch.rsqrt(bn.running_var + bn.eps)
        w = conv.weight * (bn.weight * inv).view(-1, 1, 1)
        b = bn.bias - bn.weight * bn.running_mean * inv
        if conv.bias is not None:
            b = b + conv.bias * bn.weight * inv
        new = nn.Conv1d(conv.in_channels, conv.out_channels, conv.kernel_size[0],
                        stride=conv.stride[0], padding=conv.padding[0], bias=True,
                        device=w.device, dtype=w.dtype)
        new.weight.copy_(w)
        new.bias.copy_(b)
        seq[0] = new
        seq[1] = nn.Identity()
    left = [n for n, mod in m.named_modules() if isinstance(mod, nn.BatchNorm1d)]
    assert not left, f"BatchNorm survived folding at {left}; the cbr pattern changed"
    for p in m.parameters():
        p.requires_grad_(False)
    return m.eval()


def load_folded(template, model):
    """Fold `model` and copy the result into `template` in place (same folded structure).
    strict=True: a key mismatch means the template was built from a different architecture."""
    template.load_state_dict(fold_bn(model).state_dict(), strict=True)
    return template


@torch.inference_mode()
def eval_model(compiled, eager, TX, TY, cfg, want_preds=False):
    """Confusion matrix of one folded model over the whole resident test set.

    Counts stay on the device: a per-batch .item() would sync ~650 times a pass, and the
    argmax of a row of NaN is 0 -- a perfectly ordinary class index -- so non-finite logits
    are counted explicitly instead of trusted. The tail batch runs eagerly (different shape
    would recompile the graph), same weights, same math."""
    C, EB, n = cfg["num_classes"], cfg["eval_batch"], TX.shape[0]
    dev = TX.device
    cm = torch.zeros(C * C, dtype=torch.long, device=dev)
    nonfin = torch.zeros((), dtype=torch.long, device=dev)
    preds = torch.empty(n, dtype=torch.uint8, device=dev) if want_preds else None
    ac = torch.autocast("cuda", dtype=torch.float16) if dev.type == "cuda" \
        else torch.autocast("cpu", enabled=False)
    for i in range(0, n, EB):
        j = min(i + EB, n)
        m = compiled if j - i == EB else eager
        with ac:
            z = m(TX[i:j].float())
        nonfin += (~torch.isfinite(z)).sum()
        p = z.argmax(1)
        cm += torch.bincount(TY[i:j].long() * C + p, minlength=C * C)
        if want_preds:
            preds[i:j] = p.to(torch.uint8)
    return cm.view(C, C), int(nonfin.item()), preds


In [ ]:
%%writefile /kaggle/working/proj/driver.py
"""One persistent worker per GPU, spawned once for the whole run.

Both GPUs hold the entire partition and the whole test set, so any client can train on
whichever GPU is free (longest-first dispatch) and evaluation splits the N client models
between the two GPUs. Each worker also holds a resident copy of EVERY client's weights
(N x 1.58 MB) and of theta, kept in sync by the driver after each round, so a train task
carries only a client id and an eval task only a list of ids.

Aggregation is re-sorted by client id so float addition order never depends on which
worker finished first, and every client re-seeds the default generator from (seed, round,
client) so its update does not depend on the schedule either. Different clients are
different models: they never form a process group.
"""
import json, math, shutil, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp

from proj.model import build_model, build_proxy, N_PARAMS_MODEL, N_PARAMS_PROXY
from proj.pfedes import (layout, flatten, unflatten_into, client_update, aggregate, amp,
                         select_clients, lr_at, _freeze)
from proj.evaluate import fold_bn, load_folded, eval_model
from proj.metrics import metrics_from_confusion, per_class_from_confusion, METRIC_KEYS
from proj import ckpt as C


def _resident(path, dev, chunk=1 << 22):
    """mmap -> GPU in chunks. A whole-array np.ascontiguousarray would materialise 5.7 GB
    of train features in host RAM per worker before the copy, and hands torch a read-only
    array. Chunking bounds the host side to `chunk` rows."""
    a = np.load(path, mmap_mode="r")
    t = torch.empty(tuple(a.shape), dtype=torch.from_numpy(np.array(a[:1])).dtype,
                    device=dev)
    for i in range(0, len(a), chunk):
        t[i:i + chunk] = torch.from_numpy(np.array(a[i:i + chunk]))
    return t


def _verdict(ref_z, got_z, ref_g, got_g, n_rows, label):
    """Decisive-row argmax agreement plus bounded logit/gradient deltas. Counts are
    integers: torch.mean on CUDA returns 0.99999994 for a perfect match."""
    dz = (got_z - ref_z).abs().max().item()
    gn = ref_g.norm().item()
    dg = (got_g - ref_g).norm().item() / (gn + 1e-12)
    flip_all = int((got_z.argmax(1) != ref_z.argmax(1)).sum())
    top2 = ref_z.topk(2, dim=1).values
    decisive = (top2[:, 0] - top2[:, 1]) > max(10 * dz, 1e-3)
    n_dec = int(decisive.sum())
    flip_dec = int((got_z.argmax(1) != ref_z.argmax(1))[decisive].sum())
    # `flip_dec == 0` over an EMPTY decisive set says nothing at all.
    if n_dec < n_rows // 10:
        raise RuntimeError(f"{label}: cannot certify, only {n_dec} of {n_rows} rows have "
                           f"a margin above {max(10 * dz, 1e-3):.2e}")
    if flip_dec or not math.isfinite(dz) or not math.isfinite(dg) or dz > 5e-2 or dg > 5e-2:
        raise RuntimeError(f"{label}: mismatch dlogit={dz} dgrad_rel={dg} "
                           f"flips {flip_dec}/{n_dec} decisive, {flip_all} of all")
    return f"max|dlogit|={dz:.2e} rel|dgrad|={dg:.2e} flips {flip_all}/{n_rows} ({flip_dec}/{n_dec} decisive)"


def _compile_train(Fe, Ge, cfg, dev, xb, yb, rank=0):
    """reduce-overhead captures forward+backward into a CUDA graph. torch.compile is lazy,
    so a try around the call catches nothing -- run both pFedES steps and compare against
    eager from the SAME state, with Dropout off on BOTH sides (Inductor functionalises RNG,
    so the two masks can never coincide; measured 6.07e-01 at p=0.1 vs 7.32e-04 at p=0).
    Warm-up captures the graphs at the production p, so restoring p reuses those entries."""
    if not cfg["compile"]:
        return Fe, Ge
    mods = list(Fe.modules()) + list(Ge.modules())
    drops = [m for m in mods if isinstance(m, nn.Dropout)]
    keep = [m.p for m in drops]
    snap = [{k: v.detach().clone() for k, v in m.state_dict().items()} for m in (Fe, Ge)]
    rng = torch.get_rng_state()
    crng = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    mu = cfg["mu"]

    def restore():
        with torch.no_grad():
            for m, s in zip((Fe, Ge), snap):
                sd = m.state_dict()
                for k, v in s.items():
                    sd[k].copy_(v)                  # copy_ keeps addresses -> graph stays valid
        torch.set_rng_state(rng)
        if crng is not None: torch.cuda.set_rng_state_all(crng)
        Fe.zero_grad(set_to_none=True); Ge.zero_grad(set_to_none=True)
        _freeze(Fe, False); _freeze(Ge, False)

    def probe(Fm, Gm):
        """Both production-shaped steps: Step 1 (F trains on [x_hat; x]) then Step 2
        (G trains through a frozen F), with the same freezing the run uses."""
        restore()
        _freeze(Ge, True); _freeze(Fe, False)
        with torch.no_grad(), amp(cfg):
            xhat = Gm(xb)
        with amp(cfg):
            z = Fm(torch.cat([xhat.float(), xb], dim=0))
        z = z.float(); n = xb.shape[0]
        (mu * F.cross_entropy(z[:n], yb) + (1 - mu) * F.cross_entropy(z[n:], yb)).backward()
        z1 = z.clone()
        g1 = torch.cat([p.grad.reshape(-1).float().clone() for p in Fe.parameters()])
        Fe.zero_grad(set_to_none=True)
        _freeze(Fe, True); _freeze(Ge, False)
        with amp(cfg):
            z2 = Fm(Gm(xb).float())
        z2 = z2.float()
        F.cross_entropy(z2, yb).backward()
        z2 = z2.clone()
        g2 = torch.cat([p.grad.reshape(-1).float().clone() for p in Ge.parameters()])
        Ge.zero_grad(set_to_none=True)
        _freeze(Fe, False); _freeze(Ge, False)
        return z1, g1, z2, g2

    try:
        Fc = torch.compile(Fe, mode="reduce-overhead")     # CUDA graphs: the 2.9x on T4
        Gc = torch.compile(Ge, mode="reduce-overhead")
        for _ in range(3):                                  # warm up + capture at production p
            probe(Fc, Gc)
        for m in drops: m.p = 0.0
        try:
            ref = probe(Fe, Ge)
            got = probe(Fc, Gc)
        finally:
            for m, p_ in zip(drops, keep): m.p = p_
        restore()
        n = 2 * xb.shape[0]
        s1 = _verdict(ref[0], got[0], ref[1], got[1], n, "step1")
        s2 = _verdict(ref[2], got[2], ref[3], got[3], xb.shape[0], "step2")
        print(f"[rank{rank}] compile OK | step1 {s1} | step2 {s2}", flush=True)
        return Fc, Gc
    except Exception as e:                        # sm_75 Triton is the documented risk
        for m, p_ in zip(drops, keep): m.p = p_   # never leave the model with dropout off
        restore()
        print(f"[rank{rank}] compile DISABLED -> eager: {e}", flush=True)
        return Fe, Ge


def _compile_eval(Te, cfg, dev, xt, rank=0):
    """The folded eval template, compiled at the fixed eval batch. Certified against the
    eager folded template on real test rows: fp16 cannot be bit-equal, so the criterion is
    the decisive-row argmax rule with a delta ceiling."""
    if not cfg["compile"]:
        return Te
    try:
        Tc = torch.compile(Te, mode="reduce-overhead")
        with torch.inference_mode():
            for _ in range(3):
                with amp(cfg):
                    Tc(xt).float()
            with amp(cfg):
                ref = Te(xt).float().clone()
                got = Tc(xt).float().clone()
        z = torch.zeros(1, device=dev)
        s = _verdict(ref, got, z, z, xt.shape[0], "eval")
        print(f"[rank{rank}] eval compile OK | {s}", flush=True)
        return Tc
    except Exception as e:
        print(f"[rank{rank}] eval compile DISABLED -> eager: {e}", flush=True)
        return Te


def worker(rank, cfg, task_q, res_q):
    """Wrapper: a worker that dies silently leaves the parent with only an exit code, and
    the real error is always in the CHILD traceback, not the spawn wrapper."""
    try:
        _worker(rank, cfg, task_q, res_q)
    except Exception:
        import traceback
        res_q.put(("error", rank, traceback.format_exc()))
        raise


def _worker(rank, cfg, task_q, res_q):
    cuda = cfg.get("device", "cuda") == "cuda"
    dev = torch.device(f"cuda:{rank}" if cuda else "cpu")
    if cuda:
        torch.cuda.set_device(dev)
        torch.backends.cudnn.benchmark = True
    # F, G and the folded eval template share ONE code object (DAGSNet.forward), and Dynamo
    # caches per code object: 4 F variants (train/eval x dropout on/off for the gate) + 4 G
    # variants + the eval template = 9 > the default recompile limit of 8. Past the limit
    # Dynamo runs the new variant EAGERLY without raising, so the eval template would
    # silently lose CUDA graphs (measured locally: "compiled" eval 0.7x eager).
    for name in ("recompile_limit", "cache_size_limit"):
        if hasattr(torch._dynamo.config, name):
            setattr(torch._dynamo.config, name, 64)
    torch.manual_seed(cfg["seed"] + rank)
    cache = Path(cfg["cache"])
    X = _resident(cache / "train_X.f16.npy", dev)
    Y = _resident(cache / "train_y.u8.npy", dev)
    TX = _resident(cache / "test_X.f16.npy", dev)
    TY = _resident(cache / "test_y.u8.npy", dev)

    Fe = build_model(cfg).to(dev).train()
    Ge = build_proxy(cfg).to(dev).train()
    fk, ik, nP = layout(Fe)
    gk, gik, nG = layout(Ge)
    assert nP == N_PARAMS_MODEL and nG == N_PARAMS_PROXY, (nP, nG)
    B = cfg["batch"]
    # Probe on real rows: random N(0,1) has none of the heavy tails of the z-scored
    # features, and a kernel that is wrong only at large magnitude would pass on noise.
    Fc, Gc = _compile_train(Fe, Ge, cfg, dev, X[:B].float(), Y[:B].long(), rank)
    Te = fold_bn(build_model(cfg).to(dev))            # folded STRUCTURE; weights per client
    Tc = _compile_eval(Te, cfg, dev, TX[:cfg["eval_batch"]].float(), rank)
    scaler_probe = torch.amp.GradScaler("cuda", enabled=cuda)
    if cuda:
        scaler_probe.scale(torch.zeros(1, device=dev))  # force _scale to exist
        assert scaler_probe._scale is not None, "GradScaler._scale gone; skips would read as 0"
    res_q.put(("ready", rank, "eager" if Fc is Fe else "compiled",
               "eager" if Tc is Te else "compiled"))

    CW = CI = GW = GI = None
    while True:
        task = task_q.get()
        kind = task[0]
        if kind == "stop":
            return
        if kind == "backend":
            # Both ranks gate independently, so one can compile and the other fall back.
            # A round whose clients were trained on two different backends is not a round
            # anyone can reproduce; the driver forces the lower common denominator.
            if task[1] == "eager": Fc, Gc = Fe, Ge
            if task[2] == "eager": Tc = Te
            res_q.put(("backend_ok", rank, "eager" if Fc is Fe else "compiled",
                       "eager" if Tc is Te else "compiled"))
            continue
        # Payloads cross the process boundary as numpy arrays: a torch tensor on a
        # multiprocessing queue is shared through /dev/shm, which a container may cap at
        # 64 MB, and the resident client table is 160 MB at 100 clients.
        if kind == "init":
            CW, CI = torch.from_numpy(task[1]).to(dev), torch.from_numpy(task[2]).to(dev)
            GW, GI = torch.from_numpy(task[3]).to(dev), torch.from_numpy(task[4]).to(dev)
            res_q.put(("init_ok", rank))
            continue
        if kind == "set_clients":
            cids = task[1]
            fvs, ivs = torch.from_numpy(task[2]).to(dev), torch.from_numpy(task[3]).to(dev)
            for j, c in enumerate(cids):
                CW[c].copy_(fvs[j]); CI[c].copy_(ivs[j])
            res_q.put(("set_ok", rank))
            continue
        if kind == "set_global":
            GW.copy_(torch.from_numpy(task[1]).to(dev)); GI.copy_(torch.from_numpy(task[2]).to(dev))
            res_q.put(("set_ok", rank))
            continue
        if kind == "eval":
            cids, want_preds = task[1], task[2]
            t0 = time.monotonic()
            if cuda: torch.cuda.reset_peak_memory_stats(dev)
            cms, nfs, preds = [], [], []
            for c in cids:
                unflatten_into(Fe, CW[c], CI[c], fk, ik)
                load_folded(Te, Fe)
                cm, nf, p = eval_model(Tc, Te, TX, TY, cfg, want_preds)
                cms.append(cm.cpu()); nfs.append(nf)
                if want_preds: preds.append(p.cpu())
            res_q.put(("eval", rank, list(cids),
                       torch.stack(cms).numpy() if cms else None, nfs,
                       torch.stack(preds).numpy() if preds else None,
                       (torch.cuda.max_memory_allocated(dev) / 2**30) if cuda else 0.0,
                       time.monotonic() - t0))
            continue
        if kind == "train":
            cid, lo, hi, rnd = task[1], task[2], task[3], task[4]
            t0 = time.monotonic()
            if cuda: torch.cuda.reset_peak_memory_stats(dev)
            unflatten_into(Fe, CW[cid], CI[cid], fk, ik)      # w_k^{t-1}
            unflatten_into(Ge, GW, GI, gk, gik)               # theta^{t-1} from the server
            # AdamW re-created per client per round (owner's decision): theta arrives fresh
            # from the server, and F_k's moments are reset with it. fused=True collapses
            # the step into one multi-tensor kernel. The rate is the round's value of the
            # schedule (proj.pfedes.lr_at), the same for eta_w and eta_theta.
            lr = lr_at(cfg, rnd)
            optF = torch.optim.AdamW(Fe.parameters(), lr=lr,
                                     weight_decay=cfg["weight_decay"], fused=cuda)
            optG = torch.optim.AdamW(Ge.parameters(), lr=lr,
                                     weight_decay=cfg["weight_decay"], fused=cuda)
            scF = torch.amp.GradScaler("cuda", enabled=cuda)
            scG = torch.amp.GradScaler("cuda", enabled=cuda)
            if cuda:
                scF.scale(torch.zeros(1, device=dev)); scG.scale(torch.zeros(1, device=dev))
            # Every stochastic input to this client derives from (seed, round, client):
            # the default generator drives Dropout, `g` drives the shuffles.
            s = cfg["seed"] * 1_000_003 + rnd * 10_007 + cid
            torch.manual_seed(s)
            g = torch.Generator(device=dev); g.manual_seed(s)
            a1, n1, a2, n2 = client_update(Fc, Fe, Gc, Ge, optF, optG, scF, scG,
                                           X, Y, lo, hi, cfg, g)
            fv, iv = flatten(Fe, fk, ik)
            gfv, giv = flatten(Ge, gk, gik)
            sk1, sk2 = int(a1["skips"].item()), int(a2["skips"].item())
            ap1, ap2 = n1 - sk1, n2 - sk2                    # NOT max(1, .): 0 must stay 0
            d1, d2 = max(1, ap1), max(1, ap2)
            res_q.put(("train", cid, hi - lo, fv.cpu().numpy(), iv.cpu().numpy(),
                       gfv.cpu().numpy(), giv.cpu().numpy(),
                       {"round": rnd, "cid": cid, "n_k": hi - lo, "rank": rank, "seed": s,
                        "lr": lr, "steps_w": n1, "applied_w": ap1, "skipped_w": sk1,
                        "nonfinite_w": int(a1["nonfin"].item()),
                        "loss_w": float(a1["loss"]) / d1,          # Eq. (6), applied steps
                        "ce_orig": float(a1["extra"]) / d1,        # CE on original x only
                        "gnorm_w": float(a1["gn"]) / d1,
                        "steps_theta": n2, "applied_theta": ap2, "skipped_theta": sk2,
                        "nonfinite_theta": int(a2["nonfin"].item()),
                        "loss_theta": float(a2["loss"]) / d2,      # Eq. (9), applied steps
                        "gnorm_theta": float(a2["gn"]) / d2,
                        "sec": time.monotonic() - t0,
                        "vram_gb": (torch.cuda.max_memory_allocated(dev) / 2**30
                                    if cuda else 0.0)}))


def check_updates(rnd, results, stats, selected, max_skips=None):
    """Every reason a round must not be aggregated, in one pure function so it can be
    tested without two GPUs and a spawned worker.

    Skipped steps are NOT a failure: each client starts fresh GradScalers at 2**16 and
    spends a few steps calibrating. What must be rejected is a client that applied no step
    in either phase, one that skipped far more than calibration explains, one whose
    APPLIED steps carried a non-finite gradient, and non-finite weights."""
    if sorted(stats) != sorted(selected):
        raise RuntimeError(f"round {rnd}: reported {sorted(stats)}, selected {sorted(selected)}")
    bad = [cid for cid, _, fv, _, gfv, _, _ in results
           if not (np.isfinite(fv).all() and np.isfinite(gfv).all())]
    if bad:
        raise RuntimeError(f"round {rnd}: non-finite weights from clients {bad}")
    for c in sorted(stats):
        st = stats[c]
        for ph in ("w", "theta"):
            if st[f"applied_{ph}"] + st[f"skipped_{ph}"] != st[f"steps_{ph}"]:
                raise RuntimeError(f"round {rnd}: client {c} applied+skipped != steps ({ph})")
    for ph in ("w", "theta"):
        dead = [c for c in sorted(stats) if stats[c][f"applied_{ph}"] == 0]
        if dead:
            raise RuntimeError(f"round {rnd}: clients {dead} applied zero {ph} steps; "
                               "they would contribute unchanged weights")
        diverged = [c for c in sorted(stats) if stats[c][f"nonfinite_{ph}"]]
        if diverged:
            raise RuntimeError(f"round {rnd}: clients {diverged} APPLIED a {ph} step whose "
                               "gradient was not finite")
        if max_skips is not None:
            over = [c for c in sorted(stats) if stats[c][f"skipped_{ph}"] > max_skips]
            if over:
                raise RuntimeError(
                    f"round {rnd}: clients {over} skipped more {ph} steps than the warm-up "
                    f"budget ({max_skips}): "
                    + ", ".join(f"{c}={stats[c][f'skipped_{ph}']}" for c in over))


def _collect(res_q, procs, n, timeout=7200):
    """A worker killed by the OS puts nothing on the queue. Poll in short slices and check
    liveness between them, or an OOM kill becomes a multi-hour hang."""
    out, deadline = [], time.time() + timeout
    while len(out) < n:
        try:
            msg = res_q.get(timeout=2.0)
            if msg[0] == "error":
                raise RuntimeError(f"worker {msg[1]} raised:\n{msg[2]}")
            out.append(msg)
        except RuntimeError:
            raise
        except Exception:
            for p in procs:
                if not p.is_alive() and p.exitcode not in (0, None):
                    raise RuntimeError(f"worker {p.pid} died, exitcode {p.exitcode} "
                                       f"(negative = signal; -9 is the OOM killer)")
            if time.time() > deadline:
                raise RuntimeError(f"timed out waiting for {n - len(out)} results")
    return out


def _shutdown(procs, task_qs):
    for q in task_qs:
        try: q.put(("stop",))
        except Exception: pass
    for p in procs:
        p.join(timeout=60)
        if p.is_alive():
            p.terminate(); p.join(timeout=10)


def _broadcast(task_qs, res_q, procs, msg):
    for q in task_qs: q.put(msg)
    return _collect(res_q, procs, len(task_qs))


def run(cfg, spans, class_names, wandb_run=None, t_origin=None):
    """t_origin is a time.monotonic() reading from when the SESSION started, not from when
    this call did. Worker spawn, the resident copy and compilation are minutes the 12 h cap
    charges for, and a deadline that started here would happily begin a round the session
    cannot finish."""
    t_start = t_origin if t_origin is not None else time.monotonic()
    mp.set_start_method("spawn", force=True)
    ctx = mp.get_context("spawn")
    task_qs = [ctx.Queue() for _ in range(cfg["world_size"])]
    res_q = ctx.Queue()
    procs = [ctx.Process(target=worker, args=(r, cfg, task_qs[r], res_q), daemon=True)
             for r in range(cfg["world_size"])]
    try:
        for p in procs: p.start()
        ready = _collect(res_q, procs, cfg["world_size"], timeout=3600)
        bt = {m[1]: m[2] for m in ready}; be = {m[1]: m[3] for m in ready}
        if len(set(bt.values())) > 1 or len(set(be.values())) > 1:
            print(f"[driver] ranks disagree on backend train={bt} eval={be}; forcing eager",
                  flush=True)
            force = ("backend", "eager" if len(set(bt.values())) > 1 else "keep",
                     "eager" if len(set(be.values())) > 1 else "keep")
            acks = _broadcast(task_qs, res_q, procs, force)
            bt = {m[1]: m[2] for m in acks}; be = {m[1]: m[3] for m in acks}
        cfg["backend"] = sorted(set(bt.values()))[0]
        cfg["backend_eval"] = sorted(set(be.values()))[0]
        startup = time.monotonic() - t_start
        print(f"[driver] {cfg['world_size']} workers ready: train {cfg['backend']}, "
              f"eval {cfg['backend_eval']} ({startup:.0f}s into the session)", flush=True)
        cfg["startup_seconds"] = startup
        # Push the effective backend somewhere READABLE WHILE THE RUN IS ALIVE: a running
        # Kaggle kernel's stdout cannot be downloaded.
        if wandb_run is not None:
            try:
                wandb_run.config.update({"backend": cfg["backend"],
                                         "backend_eval": cfg["backend_eval"],
                                         "startup_seconds": round(startup, 1)},
                                        allow_val_change=True)
                wandb_run.summary["backend"] = cfg["backend"]
                wandb_run.summary["backend_eval"] = cfg["backend_eval"]
            except Exception as e:
                print(f"[driver] could not publish backend to W&B: {e}", flush=True)
        return _rounds(cfg, spans, class_names, wandb_run, t_start, procs, task_qs, res_q)
    finally:
        # Without this a driver-side exception leaves two processes holding both GPUs, and
        # the next cell in the notebook fails with a CUDA OOM that names nothing.
        _shutdown(procs, task_qs)


def _stats(values):
    v = np.asarray(values, dtype=np.float64)
    return float(v.mean()), float(v.std()), float(v.min()), float(v.max())


def _rounds(cfg, spans, class_names, wandb_run, t_start, procs, task_qs, res_q):
    d = C.run_dir(cfg["run_name"])
    N, W = cfg["n_clients"], cfg["world_size"]
    torch.manual_seed(cfg["seed"])                 # seed BEFORE building: w^0, theta^0 seeded
    F0, G0 = build_model(cfg), build_proxy(cfg)
    fk, ik, nP = layout(F0)
    gk, gik, nG = layout(G0)
    f0v, f0i = flatten(F0, fk, ik)
    GW, GI = flatten(G0, gk, gik)
    # Every client starts from the SAME seeded initialization (the paper does not specify
    # per-client initialization in the model-homogeneous setting).
    CW = f0v.unsqueeze(0).repeat(N, 1).contiguous()
    CI = f0i.unsqueeze(0).repeat(N, 1).contiguous()

    start = 1
    last = C.resolve_resume(cfg["run_name"], cfg)
    if last is not None:
        G, Fs, _ = C.load_weights(d / "weights" / f"round_{last:03d}.pt", build_model,
                                  build_proxy, N_PARAMS_MODEL, N_PARAMS_PROXY)
        GW, GI = flatten(G, gk, gik)
        for c, m in Fs.items():
            CW[c], CI[c] = flatten(m, fk, ik)
        start = last + 1
        print(f"[driver] resumed at round {start}")
    elif cfg.get("require_resume"):
        raise SystemExit("require_resume set and no checkpoint found")
    if start > cfg["rounds"]:
        print(f"[driver] nothing to do: {last} rounds already complete")
        return []
    _broadcast(task_qs, res_q, procs, ("init", CW.numpy(), CI.numpy(), GW.numpy(), GI.numpy()))

    n_test = cfg["n_test"]
    preds_rounds = set(cfg.get("preds_rounds", [cfg["rounds"]]))
    # A client that was not selected this round still holds w_k^{t-1}: its model did not
    # change, so its confusion matrix on the fixed test set is the one already on disk.
    # Re-evaluating it would spend 36 s per client per GPU to reproduce a known integer
    # matrix. The cache is exact by construction; every `eval_all_every` rounds and at the
    # last round every client is re-evaluated anyway and the cache is checked against it.
    cm_cache = {}
    if last is not None:
        prev = np.load(d / "confusion" / f"round_{last:03d}.npy")
        cm_cache = {c: prev[c] for c in range(N)}
    every = int(cfg.get("eval_all_every", 10))
    hist = []
    reserve = cfg.get("finalize_reserve_seconds", 600)
    elapsed = time.monotonic() - t_start
    if elapsed + reserve >= cfg["max_seconds"]:
        print(f"[driver] no round started: {elapsed/3600:.2f} h of the "
              f"{cfg['max_seconds']/3600:.2f} h budget is already gone", flush=True)
        return hist

    for rnd in range(start, cfg["rounds"] + 1):
        t0 = time.monotonic()
        sel = select_clients(N, cfg["participation"], cfg["seed"], rnd)
        # longest-first bounds the idle tail: sending the biggest client last strands a GPU
        order = sorted(sel, key=lambda c: spans[c][1] - spans[c][0], reverse=True)
        pending, nxt, results = {}, 0, []
        for r in range(W):                                  # prime both GPUs
            if nxt < len(order):
                c = order[nxt]; nxt += 1
                task_qs[r].put(("train", c, *spans[c], rnd)); pending[r] = c
        while len(results) < len(order):
            msg = _collect(res_q, procs, 1)[0]
            assert msg[0] == "train", msg[0]
            results.append(msg[1:])
            r = next(k for k, v in pending.items() if v == msg[1])
            if nxt < len(order):
                c = order[nxt]; nxt += 1
                task_qs[r].put(("train", c, *spans[c], rnd)); pending[r] = c
            else:
                pending.pop(r)
        t_train = time.monotonic() - t0

        results.sort(key=lambda t: t[0])                    # NOT completion order
        stats = {cid: s for cid, _, _, _, _, _, s in results}
        check_updates(rnd, results, stats, sel, cfg.get("max_skips_per_client"))
        GW, GI = aggregate([(cid, nk, torch.from_numpy(gfv), torch.from_numpy(giv))
                            for cid, nk, _, _, gfv, giv, _ in results])
        cids = [t[0] for t in results]
        fvs = np.stack([t[2] for t in results]); ivs = np.stack([t[3] for t in results])
        for j, c in enumerate(cids):
            CW[c].copy_(torch.from_numpy(fvs[j])); CI[c].copy_(torch.from_numpy(ivs[j]))
        _broadcast(task_qs, res_q, procs, ("set_clients", cids, fvs, ivs))
        _broadcast(task_qs, res_q, procs, ("set_global", GW.numpy(), GI.numpy()))

        # ---- evaluate the personalized models F_k(x) on the full test set: every client
        # whose weights changed, plus everyone on a full-eval round
        want_preds = rnd in preds_rounds
        full_eval = (rnd % every == 0) or rnd == cfg["rounds"] or want_preds \
            or any(c not in cm_cache for c in range(N))
        to_eval = list(range(N)) if full_eval else list(sel)
        # Client c is ALWAYS evaluated on worker c % W. The two workers are separate
        # processes on separate GPUs, and cuDNN's benchmark-mode algorithm choice is made
        # per process: measured in the 100-client run, every client whose partial-round
        # eval had landed on the other GPU (28 of 90) came back with a confusion matrix a
        # few rows off at the round-10 re-check, while every same-GPU client matched
        # exactly. Splitting by position balanced the queues but made the cache check
        # compare two GPUs; splitting by client id compares a worker with itself.
        eval_split = [[c for c in to_eval if c % W == r] for r in range(W)]
        t1 = time.monotonic()
        for r in range(W):
            task_qs[r].put(("eval", eval_split[r], want_preds))
        ev = _collect(res_q, procs, W)
        t_eval = time.monotonic() - t1
        fresh = {}
        preds = np.empty((N, n_test), dtype=np.uint8) if want_preds else None
        nf_total = 0
        for e in ev:
            for j, c in enumerate(e[2]):
                fresh[c] = e[3][j]; nf_total += e[4][j]
                if want_preds: preds[c] = e[5][j]
        assert sorted(fresh) == sorted(to_eval), f"evaluated {sorted(fresh)}"
        if nf_total:
            raise RuntimeError(f"round {rnd}: {nf_total} non-finite test logits; argmax "
                               "would have turned them into ordinary class labels")
        # The cache check: an unselected client re-evaluated on a full-eval round must
        # reproduce its cached matrix exactly. A mismatch is not fatal -- the fresh value
        # wins -- but it is recorded, because it would mean the eval is not deterministic.
        mismatch = [c for c in fresh if c not in sel and c in cm_cache
                    and not np.array_equal(fresh[c], cm_cache[c])]
        if mismatch:
            print(f"[r{rnd:03d}] WARNING: cached confusion differs from re-evaluation for "
                  f"clients {mismatch[:8]}{'...' if len(mismatch) > 8 else ''}", flush=True)
        cm_cache.update(fresh)
        cm = np.stack([cm_cache[c] for c in range(N)])
        assert (cm.sum(axis=(1, 2)) == n_test).all(), "a confusion matrix misses test rows"
        per_client = []
        for c in range(N):
            m = metrics_from_confusion(cm[c])
            per_client.append({"cid": c, "selected": c in sel, "evaluated": c in fresh, **m,
                               "per_class": per_class_from_confusion(cm[c], class_names)})
        row = {"round": rnd, "selected": len(sel), "evaluated": len(fresh),
               "cache_mismatch": len(mismatch), "lr": lr_at(cfg, rnd)}
        for k in METRIC_KEYS:                                # mean over ALL N clients
            mean, std, lo, hi = _stats([pc[k] for pc in per_client])
            row[k] = mean
            row[f"{k}_std"], row[f"{k}_min"], row[f"{k}_max"] = std, lo, hi
        # *_client_mean is the unweighted mean ACROSS SELECTED CLIENTS of each client's
        # mean over its applied steps -- not the mean over training samples.
        row.update({
            "loss_w_client_mean": float(np.mean([s["loss_w"] for s in stats.values()])),
            "ce_orig_client_mean": float(np.mean([s["ce_orig"] for s in stats.values()])),
            "loss_theta_client_mean": float(np.mean([s["loss_theta"] for s in stats.values()])),
            "gnorm_w": float(np.mean([s["gnorm_w"] for s in stats.values()])),
            "gnorm_theta": float(np.mean([s["gnorm_theta"] for s in stats.values()])),
            "steps_w": int(sum(s["steps_w"] for s in stats.values())),
            "skipped_w": int(sum(s["skipped_w"] for s in stats.values())),
            "steps_theta": int(sum(s["steps_theta"] for s in stats.values())),
            "skipped_theta": int(sum(s["skipped_theta"] for s in stats.values())),
            "train_sec": t_train, "eval_sec": t_eval,
            "vram_train_gb": max(s["vram_gb"] for s in stats.values()),
            "vram_eval_gb": max(e[6] for e in ev),
            "backend": cfg.get("backend", "?"), "seconds": 0.0})
        mean_metrics = {k: row[k] for k in METRIC_KEYS}

        # ---- commit. Marker absolutely last.
        unflatten_into(G0, GW, GI, gk, gik)
        clients_sd = {}
        for c in range(N):
            unflatten_into(F0, CW[c], CI[c], fk, ik)
            clients_sd[c] = C.cpu_sd(F0)
        C.save_round_weights(C.cpu_sd(G0), clients_sd, rnd, cfg, mean_metrics, d)
        C.atomic_np_save(d / "confusion" / f"round_{rnd:03d}.npy", cm)
        if want_preds:
            C.atomic_np_save(d / "preds" / f"round_{rnd:03d}.u8.npy", preds)
        (d / "logs" / f"round_{rnd:03d}.json").write_text(
            json.dumps({"selected": sel, "clients": [stats[c] for c in sorted(stats)]},
                       indent=1))
        # `seconds` BEFORE the W&B call and the JSON: the budget below compares absolute
        # session elapsed, so the commit tail lands in the next round's elapsed and the
        # finalize reserve covers the last one.
        row["seconds"] = time.monotonic() - t0
        if wandb_run is not None:
            wandb_run.log({k: v for k, v in row.items()
                           if k not in ("round", "backend")}, step=rnd)
        (d / "metrics" / f"round_{rnd:03d}.json").write_text(json.dumps(
            {**row, "clients": per_client}, indent=1))
        C.append_history(d, row, [{"round": rnd, **{k: v for k, v in pc.items()
                                                    if k != "per_class"}}
                                  for pc in per_client])
        C.mark_complete(d, rnd)
        hist.append(row)
        print(f"[r{rnd:03d}] f1_macro mean={row['f1_macro']:.6f} "
              f"std={row['f1_macro_std']:.4f} min={row['f1_macro_min']:.4f} "
              f"acc={row['accuracy']:.6f} lr={row['lr']:.2e} "
              f"loss_w={row['loss_w_client_mean']:.4f} "
              f"loss_th={row['loss_theta_client_mean']:.4f} "
              f"skip={row['skipped_w']}+{row['skipped_theta']}/{row['steps_w']}+{row['steps_theta']} "
              f"train={t_train:.0f}s eval={t_eval:.0f}s vram={row['vram_train_gb']:.2f}G "
              f"{row['seconds']:.1f}s | session {(time.monotonic()-t_start)/3600:.2f}h",
              flush=True)

        worst = max(h["seconds"] for h in hist)
        if (time.monotonic() - t_start) + worst * 1.15 + reserve > cfg["max_seconds"]:
            print(f"[driver] stopping after round {rnd}: the next round plus a "
                  f"{reserve/60:.0f} min finalize reserve would exceed the session budget "
                  f"({cfg['max_seconds']/3600:.2f} h)", flush=True)
            break

    return hist


def write_manifest(cfg, class_names, spans, y_true_src=None, extra=None):
    """Everything needed to say what these numbers are, written once, next to them.

    Also the SECOND data gate: `content_id` is computed after the decode from the row
    counts and the class histogram and compared against what the resumed checkpoint was
    trained on. Continuing on top of different data is not a warning."""
    d = C.run_dir(cfg["run_name"])
    mf = d / "reports" / "manifest.json"
    m = {"fingerprint": C.fingerprint(cfg),
         "cfg": {k: v for k, v in cfg.items()},
         "class_names": list(class_names),
         "n_clients": len(spans), "n_train": sum(h - l for l, h in spans.values()),
         "client_rows": {str(c): spans[c][1] - spans[c][0] for c in sorted(spans)},
         "n_params_model": N_PARAMS_MODEL, "n_params_proxy": N_PARAMS_PROXY,
         "torch": torch.__version__, "cuda": torch.version.cuda,
         "written": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    if extra: m.update(extra)

    if mf.is_file():
        old = json.loads(mf.read_text())
        for k in ("content_id", "data_id"):
            a, b = old.get(k), m.get(k)
            if a is not None and b is not None and a != b:
                raise RuntimeError(
                    f"{k} changed: this run's checkpoints were trained on {a}, the data "
                    f"mounted now is {b}. Resuming across that is not a continuation.")
        m["sessions"] = int(old.get("sessions", 1)) + 1
        m["first_written"] = old.get("first_written", old.get("written"))
    else:
        m["sessions"], m["first_written"] = 1, m["written"]

    # y_true travels with the run: a downloaded run directory must be able to check its
    # own predictions against its own confusion matrices.
    if y_true_src is not None:
        dst = d / "reports" / "y_true.u8.npy"
        if not dst.is_file():
            shutil.copyfile(y_true_src, dst)
        m["y_true"] = "reports/y_true.u8.npy"
    mf.write_text(json.dumps(m, indent=2))
    return mf


In [ ]:
%%writefile /kaggle/working/proj/verify.py
"""Re-derive every published number from the artifacts on disk.

Nothing here trusts a number because it was printed once. Per client and per round, the
10 metrics and the per-class block are recomputed from that client's confusion matrix; the
mean/std/min/max over clients are recomputed from the per-client values; predictions, where
stored, rebuild the confusion matrices; the client logs are checked against the step
arithmetic and the seeded selection they claim; and all of it is compared against the
copies in the weights file, history.csv and clients.csv.

Every check exists because its absence lets a specific tampered fixture pass: a deleted
history row, a duplicated one, a metric set to NaN (`abs(nan) > tol` is False), a per-class
F1 of 999, deleted logs, a client dropped from the selection, a resume file overwritten
with garbage, and a config claiming 999 test rows.
"""
import csv, json, math
from pathlib import Path
import numpy as np
import torch

from proj.metrics import metrics_from_confusion, per_class_from_confusion, METRIC_KEYS
from proj.pfedes import select_clients, expected_steps, lr_at
from proj import ckpt as C

TOL = 1e-12          # both sides come from the same float64 code path on the same counts
CSV_TOL = 1e-9       # history.csv round-trips through str()
# A client re-evaluated with unchanged weights must reproduce its cached confusion matrix.
# Within one worker the eval is bit-exact (0 differing rows over ~250 same-GPU re-checks in
# the 50c/100c runs of 2026-09-11). Across the two GPUs -- separate processes, cuDNN
# benchmark-mode algorithm choice per process -- fp16 logits differ on a handful of rows:
# measured at most 23 of 10,761,343 (2e-6), max |delta metric| 1.9e-5. The driver now pins
# client c to worker c % world_size, so new runs re-check on the same GPU; the runs made
# before that pin are accepted through this bound, which is 4x the worst measured noise
# and orders of magnitude below what one training step moves (thousands of rows).
CACHE_TOL_ROWS = 100


def _finite(x):
    try: return math.isfinite(float(x))
    except (TypeError, ValueError): return False


def _read_csv(p):
    with open(p) as f:
        return list(csv.DictReader(f))


def verify_run(run_dir, cfg=None, build_model=None, build_proxy=None, expect_model=None,
               expect_proxy=None, y_true_path=None, require_rounds=None, full=True):
    """Returns (ok, lines). full=True is the acceptance mode: client logs must be present
    for every round and predictions for every round that claims them."""
    d = Path(run_dir)
    fp = C.fingerprint(cfg) if cfg is not None else None
    last = C.last_complete_round(d, fp) or 0
    mode = "full" if full else "minimal"
    out = [f"run      : {d}", f"complete : rounds 1..{last}   (mode: {mode})"]
    bad = []
    cache_moved = []                # (round, cid, rows moved) for tolerated cache deviations

    mf = d / "reports" / "manifest.json"
    man = json.loads(mf.read_text()) if mf.is_file() else None
    if man is None and full:
        bad.append("reports/manifest.json missing: the run does not describe itself")

    N = int(cfg["n_clients"]) if cfg else (int(man["n_clients"]) if man else None)
    n_test = int(cfg["n_test"]) if cfg and "n_test" in cfg else None
    want_rows = ({int(k): int(v) for k, v in man["client_rows"].items()}
                 if man and "client_rows" in man else None)

    # ---- history.csv / clients.csv: exactly rounds 1..last, once each
    hist, hp = {}, d / "history.csv"
    if hp.is_file():
        rows = _read_csv(hp)
        seen = [int(r["round"]) for r in rows]
        if len(seen) != len(set(seen)):
            bad.append(f"history.csv has duplicate rows for round(s) "
                       f"{sorted({r for r in seen if seen.count(r) > 1})}")
        if sorted(set(seen)) != list(range(1, last + 1)):
            bad.append(f"history.csv covers rounds {sorted(set(seen))}, expected 1..{last}")
        hist = {int(r["round"]): r for r in rows}
    elif last:
        bad.append("history.csv missing")
    crows, cp = {}, d / "clients.csv"
    if cp.is_file():
        for r in _read_csv(cp):
            crows.setdefault(int(r["round"]), {})[int(r["cid"])] = r
    elif last:
        bad.append("clients.csv missing")

    y_true = None
    if y_true_path is not None and Path(y_true_path).is_file():
        y_true = np.load(y_true_path, mmap_mode="r")

    for r in range(1, last + 1):
        tag = f"round {r:03d}"
        cm = np.load(d / "confusion" / f"round_{r:03d}.npy")
        if cm.ndim != 3 or cm.shape[1] != cm.shape[2] or (N is not None and cm.shape[0] != N):
            bad.append(f"{tag}: confusion is {cm.shape}, expected ({N}, C, C)"); continue
        if (cm < 0).any():
            bad.append(f"{tag}: negative counts in a confusion matrix")
        if cfg and cm.shape[1] != int(cfg["num_classes"]):
            bad.append(f"{tag}: {cm.shape[1]} classes, cfg says {cfg['num_classes']}")
        totals = cm.sum(axis=(1, 2))
        if n_test is None: n_test = int(totals[0])
        if not (totals == n_test).all():
            bad.append(f"{tag}: confusion totals {sorted(set(totals.tolist()))} != n_test {n_test}")

        js = json.loads((d / "metrics" / f"round_{r:03d}.json").read_text())
        ck = torch.load(d / "weights" / f"round_{r:03d}.pt", map_location="cpu",
                        weights_only=True, mmap=True)
        clients = js.get("clients", [])
        if [c.get("cid") for c in clients] != list(range(cm.shape[0])):
            bad.append(f"{tag}: metrics json lists clients "
                       f"{[c.get('cid') for c in clients][:5]}..., expected 0..{cm.shape[0]-1}")
            continue
        per = {k: [] for k in METRIC_KEYS}
        prev_cm = np.load(d / "confusion" / f"round_{r - 1:03d}.npy") if r > 1 else None
        n_fresh = 0
        n_mismatch = 0                  # unselected client re-evaluated: must reproduce its cache
        for c in clients:
            cid = int(c["cid"])
            if c.get("evaluated"):
                n_fresh += 1
                if prev_cm is not None and not c.get("selected") \
                        and not np.array_equal(cm[cid], prev_cm[cid]):
                    n_mismatch += 1
                    moved = int(np.abs(cm[cid] - prev_cm[cid]).sum() // 2)
                    cache_moved.append((r, cid, moved))
                    if moved > CACHE_TOL_ROWS:
                        bad.append(f"{tag}: client {cid} re-evaluated with unchanged weights but "
                                   f"its confusion moved {moved} rows (> {CACHE_TOL_ROWS}): not "
                                   "cross-GPU noise -- weights changed outside selection or "
                                   "the eval is not deterministic")
            elif prev_cm is None:
                bad.append(f"{tag}: client {cid} not evaluated in round 1")
            elif c.get("selected"):
                bad.append(f"{tag}: client {cid} was selected (weights changed) but not evaluated")
            elif not np.array_equal(cm[cid], prev_cm[cid]):
                bad.append(f"{tag}: client {cid} carried forward but its confusion differs "
                           "from the previous round's")
            rec = metrics_from_confusion(cm[cid])
            for k in METRIC_KEYS:
                v = c.get(k)
                if not _finite(v) or abs(float(v) - rec[k]) > TOL:
                    bad.append(f"{tag}: client {cid} {k} {v!r} != {rec[k]} recomputed")
                per[k].append(rec[k])
                cv = crows.get(r, {}).get(cid, {}).get(k)
                if crows and (cv is None or not _finite(cv) or abs(float(cv) - rec[k]) > CSV_TOL):
                    bad.append(f"{tag}: clients.csv client {cid} {k} = {cv!r} != {rec[k]}")
            names = [e.get("class") for e in c.get("per_class", [])]
            want_pc = per_class_from_confusion(cm[cid], names) if len(names) == cm.shape[1] else None
            if want_pc is None:
                bad.append(f"{tag}: client {cid} per-class block missing or wrong length")
            else:
                for got, want in zip(c["per_class"], want_pc):
                    for f in ("idx", "support"):
                        if int(got.get(f, -1)) != int(want[f]):
                            bad.append(f"{tag}: client {cid} class {want['idx']} {f} "
                                       f"{got.get(f)} != {want[f]}")
                    for f in ("precision", "recall", "f1"):
                        v = got.get(f)
                        if not _finite(v) or abs(float(v) - want[f]) > TOL:
                            bad.append(f"{tag}: client {cid} class {want['idx']} {f} "
                                       f"{v!r} != {want[f]}")
        # ---- the aggregate row: mean/std/min/max over clients, in json, weights, csv
        for k in METRIC_KEYS:
            v = np.asarray(per[k], dtype=np.float64)
            want = {k: float(v.mean()), f"{k}_std": float(v.std()),
                    f"{k}_min": float(v.min()), f"{k}_max": float(v.max())}
            for kk, wv in want.items():
                for where, val, tol in (("json", js.get(kk), TOL),
                                        ("history.csv", hist.get(r, {}).get(kk), CSV_TOL)):
                    if val is None:
                        bad.append(f"{tag}: {kk} missing in {where}"); continue
                    if not _finite(val):
                        bad.append(f"{tag}: {kk} in {where} is not finite ({val!r})")
                    elif abs(float(val) - wv) > tol:
                        bad.append(f"{tag}: {kk} in {where} = {val} != {wv} recomputed")
            wv = ck.get("metrics", {}).get(k)
            if wv is None or not _finite(wv) or abs(float(wv) - want[k]) > TOL:
                bad.append(f"{tag}: weights file {k} = {wv!r} != {want[k]}")
        if cfg is not None:
            k_want = len(select_clients(cm.shape[0], cfg["participation"], cfg["seed"], r))
            if int(js.get("selected", -1)) != k_want:
                bad.append(f"{tag}: json 'selected' {js.get('selected')} != K={k_want}")
        if int(js.get("evaluated", -1)) != n_fresh:
            bad.append(f"{tag}: json 'evaluated' {js.get('evaluated')} != {n_fresh} clients flagged")
        if int(js.get("cache_mismatch", -1)) != n_mismatch:
            bad.append(f"{tag}: json 'cache_mismatch' {js.get('cache_mismatch')} != {n_mismatch} "
                       "recomputed from the confusion matrices")

        if fp is not None and ck.get("fingerprint") != fp:
            bad.append(f"{tag}: weights fingerprint {ck.get('fingerprint')} != {fp}")
        if build_model is not None and build_proxy is not None:
            try:
                C.load_weights(d / "weights" / f"round_{r:03d}.pt", build_model, build_proxy,
                               expect_model, expect_proxy)
            except Exception as e:
                bad.append(f"{tag}: weights do not rebuild the models: {e}")

        # ---- predictions tie the matrices back to model output, where stored
        pp = d / "preds" / f"round_{r:03d}.u8.npy"
        claims = cfg is not None and r in set(cfg.get("preds_rounds", [cfg["rounds"]]))
        if pp.is_file():
            yp = np.load(pp, mmap_mode="r")
            if yp.shape != (cm.shape[0], n_test):
                bad.append(f"{tag}: predictions are {yp.shape}, expected ({cm.shape[0]}, {n_test})")
            elif y_true is not None:
                if len(y_true) != n_test:
                    bad.append(f"{tag}: y_true has {len(y_true)} rows, test has {n_test}")
                else:
                    k = cm.shape[1]
                    yt = np.asarray(y_true, np.int64) * k
                    for cid in range(cm.shape[0]):
                        rebuilt = np.bincount(yt + np.asarray(yp[cid], np.int64),
                                              minlength=k * k).reshape(k, k)
                        if not (rebuilt == cm[cid]).all():
                            bad.append(f"{tag}: client {cid} confusion != stored predictions")
            elif full:
                bad.append(f"{tag}: predictions present but no y_true to check them against")
        elif claims and full:
            bad.append(f"{tag}: cfg claims predictions for this round but none are stored")

        # ---- client logs: selection and step arithmetic have to close
        lp = d / "logs" / f"round_{r:03d}.json"
        if not lp.is_file():
            if full:
                bad.append(f"{tag}: no client log; participation is unattested")
        else:
            try: lg = json.loads(lp.read_text())
            except Exception as e:
                bad.append(f"{tag}: client log unreadable: {e}"); lg = {}
            sel = lg.get("selected", [])
            if cfg is not None:
                want_sel = select_clients(cm.shape[0], cfg["participation"], cfg["seed"], r)
                if list(sel) != want_sel:
                    bad.append(f"{tag}: log selection {sel[:6]}... != seeded draw {want_sel[:6]}...")
            got_ids = sorted(int(e["cid"]) for e in lg.get("clients", []))
            if got_ids != sorted(sel):
                bad.append(f"{tag}: log has clients {got_ids[:6]}..., selected {sorted(sel)[:6]}...")
            flags = {int(c["cid"]): bool(c.get("selected")) for c in clients}
            if any(flags.get(c) is not True for c in sel) or \
                    any(flags.get(c) for c in range(cm.shape[0]) if c not in set(sel)):
                bad.append(f"{tag}: 'selected' flags in metrics json disagree with the log")
            for e in lg.get("clients", []):
                c = e.get("cid")
                for ph in ("w", "theta"):
                    if e.get(f"applied_{ph}", 0) + e.get(f"skipped_{ph}", 0) != e.get(f"steps_{ph}", -1):
                        bad.append(f"{tag}: client {c} applied+skipped != steps ({ph})")
                    if e.get(f"applied_{ph}", 0) <= 0:
                        bad.append(f"{tag}: client {c} applied no {ph} step")
                if cfg and "n_k" in e:
                    s1, s2 = expected_steps(int(e["n_k"]), cfg)
                    if int(e.get("steps_w", -1)) != s1 or int(e.get("steps_theta", -1)) != s2:
                        bad.append(f"{tag}: client {c} ran {e.get('steps_w')}/{e.get('steps_theta')}"
                                   f" steps, expected {s1}/{s2}")
                if want_rows is not None and c in want_rows and int(e.get("n_k", -1)) != want_rows[c]:
                    bad.append(f"{tag}: client {c} trained on {e.get('n_k')} rows, "
                               f"manifest says {want_rows[c]}")
                # The rate the client actually used must be the schedule's value for this
                # round: a resumed session that planned a different horizon would otherwise
                # continue the run at a rate the fingerprint never saw.
                if cfg is not None:
                    want_lr = lr_at(cfg, r)
                    if not _finite(e.get("lr")) or abs(float(e["lr"]) - want_lr) > 1e-12:
                        bad.append(f"{tag}: client {c} trained at lr {e.get('lr')!r}, "
                                   f"schedule says {want_lr}")
                for f in ("loss_w", "ce_orig", "gnorm_w", "loss_theta", "gnorm_theta"):
                    if not _finite(e.get(f)):
                        bad.append(f"{tag}: client {c} {f} is not finite ({e.get(f)!r})")

    if cache_moved:
        w = max(cache_moved, key=lambda t: t[2])
        out.append(f"cache    : {len(cache_moved)} client-round re-checks differ from their cache "
                   f"by <= {w[2]} rows (round {w[0]} client {w[1]}; bound {CACHE_TOL_ROWS}) -- "
                   "cross-GPU fp16 noise, metrics affected below 1e-4")
    else:
        out.append("cache    : every re-evaluated unchanged client reproduced its cached matrix exactly")
    n_preds = len(list((d / "preds").glob("round_*.u8.npy"))) if (d / "preds").is_dir() else 0
    n_logs = len(list((d / "logs").glob("round_*.json"))) if (d / "logs").is_dir() else 0
    out.append(f"artifacts: {n_preds} prediction files, {n_logs} client logs"
               + ("" if y_true is not None else "   (predictions NOT cross-checked: no y_true)"))
    if require_rounds is not None and last != require_rounds:
        bad.append(f"run is INCOMPLETE: {last} of {require_rounds} rounds")
    out += [f"  FAIL {b}" for b in bad] or ["  all artifact checks passed"]
    return not bad, out


In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

try:
    _wandb_key = UserSecretsClient().get_secret("wandb_key")
except Exception as e:
    raise SystemExit(f"W&B secret unavailable: {e}")
wandb.login(key=_wandb_key, relogin=True, verify=True)
del _wandb_key
# W&B fails fast before dataset decode or model preparation. The key is never persisted.
run = wandb.init(project="pfedes-veremi", name="pfedes_20c_probe", config=CFG,
                 resume="allow", id="pfedes_20c_probe")
print("W&B:", run.url)


In [ ]:
# Cheap identity work, then the resume gate, then the decode. A continuation push must
# die at the gate, not after a two-minute parquet pass.
import hashlib, json, time, numpy as np
from pathlib import Path
from proj import ckpt as C
from proj.data import (find_root, load_clients, load_test, assert_fp16_safe,
                       cache_ok)

FL_ROOT = find_root("train/client_id=000")
CEN_ROOT = find_root("upload/test")
TEST_ROOT = CEN_ROOT / "upload/test"
SCALER = json.loads((CEN_ROOT / "upload/scaler.json").read_text())["features"]
assert len(SCALER) == 66, f"scaler has {len(SCALER)} entries, expected 66"
FEATS = ['f_rcv_pos_noise_x', 'f_rcv_pos_noise_y', 'f_rcv_spd', 'f_rcv_spd_noise', 'f_rcv_acl', 'f_rcv_acl_noise', 'f_rcv_hed_noise', 'f_snd_pos_noise_x', 'f_snd_pos_noise_y', 'f_snd_spd', 'f_snd_spd_noise', 'f_snd_acl', 'f_snd_acl_noise', 'f_snd_hed_noise', 'f_snd_dist_road_edge', 'f_rcv_x_rel', 'f_rcv_y_rel', 'f_snd_x_rel', 'f_snd_y_rel', 'f_delay_s', 'f_dx', 'f_dy', 'f_dist', 'f_bearing_sin', 'f_bearing_cos', 'f_rcv_hed_sin', 'f_rcv_hed_cos', 'f_snd_hed_sin', 'f_snd_hed_cos', 'f_hed_diff_cos', 'f_rcv_vx', 'f_rcv_vy', 'f_snd_vx', 'f_snd_vy', 'f_rel_speed', 'f_closing_speed', 'f_spd_diff', 'f_rcv_noise_mag', 'f_snd_noise_mag', 'f_first_in_session', 'f_sess_idx', 'f_sess_dt', 'f_sess_dt_send', 'f_sess_dt_skew', 'f_sess_dpos', 'f_sess_implied_spd', 'f_sess_spd_residual', 'f_sess_dspd', 'f_sess_acl_residual', 'f_sess_dhed', 'f_sess_dmsgid', 'f_sess_ddist', 'f_sess_ddre', 'f_sess_pos_pred_err', 'f_alias_age_s', 'f_rx_rate_1s', 'f_rx_rate_5s', 'f_sender_rate_1s', 'f_sender_rate_5s', 'f_sender_share_5s', 'f_rcv_profile_normal', 'f_rcv_profile_cautious', 'f_rcv_profile_aggressive', 'f_snd_profile_normal', 'f_snd_profile_cautious', 'f_snd_profile_aggressive']
CLASS_NAMES = ['benign', 'accelerationMultiplication', 'constantPositionOffset', 'constantSpeedOffset', 'dataReplay', 'dosAttack', 'feignedBraking', 'positionMirroring', 'randomPositionOffset', 'randomSpeedOffset', 'reversedHeading', 'suddenConstantSpeed', 'suddenStop', 'timeDelayAttack', 'trafficCongestionSybil', 'zeroSpeedReport']

# What the fingerprint could not otherwise see: a permuted feature order, a re-fitted
# scaler or a different partition keep every shape identical.
CFG["data_id"] = hashlib.sha256(json.dumps({
    "features": FEATS, "classes": CLASS_NAMES, "n_clients": CFG["n_clients"],
    "scaler": [[SCALER[c]["mean"], SCALER[c]["std_used"]] for c in FEATS],
}, sort_keys=True).encode()).hexdigest()[:16]
print("FL root    :", FL_ROOT)
print("test root  :", TEST_ROOT)
print("data_id    :", CFG["data_id"])
print("fingerprint:", C.fingerprint(CFG))

last = C.resolve_resume(CFG["run_name"], CFG)
if CFG["require_resume"] and last is None:
    raise SystemExit("require_resume set but no VERIFIED checkpoint found — fix the "
                     "attachment. A marker without its artifacts does not count.")
print("resume from round", last)

cache = Path(CFG["cache"]); cache.mkdir(parents=True, exist_ok=True)
MF = cache / "manifest.json"
FILES = ("train_X.f16.npy", "train_y.u8.npy", "test_X.f16.npy", "test_y.u8.npy",
         "spans.json")
want = {"data_id": CFG["data_id"], "n_clients": CFG["n_clients"],
        "fl_root": str(FL_ROOT), "test_root": str(TEST_ROOT)}

t0 = time.time()
if cache_ok(cache, want, CFG["n_clients"]):
    print("prepack cache reusable (manifest matches and every file checks out)")
else:
    for f in FILES: (cache / f).unlink(missing_ok=True)
    MF.unlink(missing_ok=True)
    X, Y, spans = load_clients(FL_ROOT, FEATS, CFG["n_clients"])
    print(f"train {X.shape} max|x|={assert_fp16_safe(X,'train'):.1f}")
    np.save(cache / "train_X.f16.npy", X); np.save(cache / "train_y.u8.npy", Y)
    json.dump({str(k): v for k, v in spans.items()}, open(cache / "spans.json", "w"))
    del X, Y
    TX, TY = load_test(TEST_ROOT, FEATS, SCALER)
    print(f"test  {TX.shape} max|x|={assert_fp16_safe(TX,'test'):.1f}")
    np.save(cache / "test_X.f16.npy", TX); np.save(cache / "test_y.u8.npy", TY)
    del TX, TY
    MF.write_text(json.dumps(want))                       # cache marker: absolutely last
    assert cache_ok(cache, want, CFG["n_clients"]), "the cache just written does not validate"

spans = {int(k): tuple(v) for k, v in json.load(open(cache / "spans.json")).items()}
CFG["n_test"] = len(np.load(cache / "test_y.u8.npy", mmap_mode="r"))
n_train = sum(h - l for l, h in spans.values())
assert n_train == 43_045_415, f"train rows {n_train} != 43,045,415"
assert CFG["n_test"] == 10_761_343, f"test rows {CFG['n_test']}"
assert len(spans) == CFG["n_clients"], f"{len(spans)} spans for {CFG['n_clients']} clients"

# content_id reads the labels that are actually cached, hit or miss: the row counts and
# the class histogram change when the partition or the file contents change.
_ytr = np.load(cache / "train_y.u8.npy", mmap_mode="r")
_yte = np.load(cache / "test_y.u8.npy", mmap_mode="r")
assert len(_ytr) == n_train, f"train X/y disagree: {len(_ytr)} labels for {n_train} rows"
CFG["content_id"] = hashlib.sha256(json.dumps({
    "clients": [[c, spans[c][0], spans[c][1]] for c in sorted(spans)],
    "train_hist": np.bincount(np.asarray(_ytr), minlength=16).tolist(),
    "test_hist": np.bincount(np.asarray(_yte), minlength=16).tolist(),
}, sort_keys=True).encode()).hexdigest()[:16]
del _ytr, _yte
print("content_id :", CFG["content_id"])
print(f"prepack {time.time()-t0:.1f}s | {n_train:,} train / {CFG['n_test']:,} test rows")


In [ ]:
# ---- PROBE ONLY: measure what sm_86 cannot tell us about the T4 before the rounds run.
# Train: one pFedES client step (both phases) eager vs compiled. Eval: one folded model over
# the FULL test set, eager vs compiled, at two batch sizes. Everything is freed afterwards
# so the two workers start with the whole GPU.
import gc, time, torch
from proj.model import build_model, build_proxy
from proj.pfedes import layout, flatten, unflatten_into, client_update
from proj.evaluate import fold_bn, load_folded, eval_model
from proj import driver as D

CAL = {}
dev = torch.device("cuda:0"); torch.cuda.set_device(dev)
torch.backends.cudnn.benchmark = True
for _n in ('recompile_limit', 'cache_size_limit'):
    if hasattr(torch._dynamo.config, _n): setattr(torch._dynamo.config, _n, 64)
cache = Path(CFG["cache"])
TX = D._resident(cache / "test_X.f16.npy", dev); TY = D._resident(cache / "test_y.u8.npy", dev)
lo, hi = spans[min(spans, key=lambda c: spans[c][1] - spans[c][0])]   # smallest client
hi = min(hi, lo + 400 * CFG["batch"])                                  # ~400 steps per phase
X = torch.from_numpy(np.load(cache / "train_X.f16.npy", mmap_mode="r")[lo:hi].copy()).to(dev)
Y = torch.from_numpy(np.load(cache / "train_y.u8.npy", mmap_mode="r")[lo:hi].copy()).to(dev)

def one_client(Fc, Fe, Gc, Ge):
    optF = torch.optim.AdamW(Fe.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"], fused=True)
    optG = torch.optim.AdamW(Ge.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"], fused=True)
    scF = torch.amp.GradScaler("cuda"); scG = torch.amp.GradScaler("cuda")
    scF.scale(torch.zeros(1, device=dev)); scG.scale(torch.zeros(1, device=dev))
    g = torch.Generator(device=dev); g.manual_seed(1)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    a1, n1, a2, n2 = client_update(Fc, Fe, Gc, Ge, optF, optG, scF, scG, X, Y, 0, hi - lo, CFG, g)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / (n1 + n2) * 1000, n1 + n2, int(a1["skips"]) + int(a2["skips"])

torch.manual_seed(CFG["seed"])
Fe = build_model(CFG).to(dev).train(); Ge = build_proxy(CFG).to(dev).train()
fk, ik, _ = layout(Fe); gk, gik, _ = layout(Ge)
f0, i0 = flatten(Fe, fk, ik); g0, gi0 = flatten(Ge, gk, gik)
ms, n, sk = one_client(Fe, Fe, Ge, Ge)
CAL["train_eager_ms_per_phase_step"] = ms
print(f"train eager   : {ms:.2f} ms per phase-step ({n} steps, {sk} skipped) at batch {CFG['batch']}")
unflatten_into(Fe, f0, i0, fk, ik); unflatten_into(Ge, g0, gi0, gk, gik)
t0 = time.perf_counter()
Fc, Gc = D._compile_train(Fe, Ge, CFG, dev, X[:CFG["batch"]].float(), Y[:CFG["batch"]].long())
CAL["compile_seconds"] = time.perf_counter() - t0
CAL["train_backend"] = "compiled" if Fc is not Fe else "eager"
if Fc is not Fe:
    unflatten_into(Fe, f0, i0, fk, ik); unflatten_into(Ge, g0, gi0, gk, gik)
    one_client(Fc, Fe, Gc, Ge)                                  # warm the graphs
    unflatten_into(Fe, f0, i0, fk, ik); unflatten_into(Ge, g0, gi0, gk, gik)
    ms, n, sk = one_client(Fc, Fe, Gc, Ge)
    CAL["train_compiled_ms_per_phase_step"] = ms
    print(f"train compiled: {ms:.2f} ms per phase-step ({n} steps, {sk} skipped) | "
          f"{CAL['train_eager_ms_per_phase_step']/ms:.2f}x | compile+gate {CAL['compile_seconds']:.0f}s")
del Fc, Gc

Te = fold_bn(build_model(CFG).to(dev)); load_folded(Te, Fe)
for eb in (8192, 16384):
    c = dict(CFG, eval_batch=eb)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    cm, nf, _ = eval_model(Te, Te, TX, TY, c); torch.cuda.synchronize()
    r = TX.shape[0] / (time.perf_counter() - t0); CAL[f"eval_eager_folded_{eb}"] = r
    print(f"eval eager-folded  batch {eb:>5}: {r:,.0f} rows/s")
    if CFG["compile"]:
        Tc = D._compile_eval(Te, c, dev, TX[:eb].float())
        if Tc is not Te:
            eval_model(Tc, Te, TX[:4 * eb], TY[:4 * eb], c)          # warm
            torch.cuda.synchronize(); t0 = time.perf_counter()
            cm2, nf2, _ = eval_model(Tc, Te, TX, TY, c); torch.cuda.synchronize()
            r = TX.shape[0] / (time.perf_counter() - t0); CAL[f"eval_compiled_folded_{eb}"] = r
            print(f"eval compiled-folded batch {eb:>5}: {r:,.0f} rows/s | "
                  f"|dCM|={int((cm2 - cm).abs().sum())} cells of {TX.shape[0]}")
            torch._dynamo.reset()
        del Tc
del Te, Fe, Ge, X, Y, TX, TY, f0, g0
gc.collect(); torch.cuda.empty_cache(); torch._dynamo.reset()
CAL["vram_after_free_gb"] = torch.cuda.memory_allocated(dev) / 2**30
print("calibration:", json.dumps({k: (round(v, 3) if isinstance(v, float) else v) for k, v in CAL.items()}))
(C.run_dir(CFG["run_name"]) / "reports" / "calibration.json").write_text(json.dumps(CAL, indent=1))
if run is not None:
    run.summary.update({f"cal_{k}": v for k, v in CAL.items()})


In [ ]:
from proj.driver import run as train, write_manifest
# Raises if this run's checkpoints were trained on different data. The resume gate above
# ran before the decode and could only compare data_id; content_id is the post-decode one.
write_manifest(CFG, CLASS_NAMES, spans,
               y_true_src=Path(CFG["cache"]) / "test_y.u8.npy",
               extra={"fl_root": str(FL_ROOT), "test_root": str(TEST_ROOT),
                      "feature_cols": FEATS, "n_test": CFG["n_test"],
                      "data_id": CFG["data_id"], "content_id": CFG["content_id"],
                      "scaler": {c: [SCALER[c]["mean"], SCALER[c]["std_used"]]
                                 for c in FEATS}})
hist = train(CFG, spans, CLASS_NAMES, wandb_run=run, t_origin=T0)
print(f"\ncompleted {len(hist)} rounds this session")


In [ ]:
# Every published number, re-derived from the artifacts on disk. Never from memory.
import csv
from proj.verify import verify_run
from proj.model import build_model, build_proxy, N_PARAMS_MODEL, N_PARAMS_PROXY
from proj.metrics import METRIC_KEYS

d = C.run_dir(CFG["run_name"])
# y_true from the RUN, not from /kaggle/temp: the cache is gone with the session, and the
# check has to be the same one someone can repeat after downloading the output alone.
ok, lines = verify_run(d, cfg=CFG, build_model=build_model, build_proxy=build_proxy,
                       expect_model=N_PARAMS_MODEL, expect_proxy=N_PARAMS_PROXY,
                       y_true_path=d / "reports" / "y_true.u8.npy", full=True)
print("\n".join(lines))

last = C.last_complete_round(d, C.fingerprint(CFG)) or 0
print(f"\nrounds verified : {last} / {CFG['rounds']}")
if last < CFG["rounds"]:
    print(f"  INCOMPLETE — attach this notebook's output (or a checkpoint dataset of it) to "
          f"the next push and regenerate with --require-resume to continue at round {last + 1}")
rows = [r for r in csv.DictReader(open(d / "history.csv")) if int(r["round"]) <= last]
if rows:
    fin = rows[-1]
    # The headline is the LAST round, fixed before the run. best-f1 is chosen on the test
    # set after seeing it, so it is a description of the curve and not a second result.
    print(f"\nresult at round {fin['round']} (mean over {CFG['n_clients']} clients; "
          f"std / min / max of f1_macro {float(fin['f1_macro_std']):.4f} / "
          f"{float(fin['f1_macro_min']):.4f} / {float(fin['f1_macro_max']):.4f}):")
    for k in METRIC_KEYS: print(f"  {k:<20} {float(fin[k]):.6f}")
    b = max(rows, key=lambda r: float(r["f1_macro"]))
    print(f"\n[descriptive only] best mean f1_macro {float(b['f1_macro']):.6f} "
          f"at round {b['round']} — picked on test, not a reported result")
    mm = sum(int(r.get("cache_mismatch", 0) or 0) for r in rows)
    print(f"eval cache re-check deviations over all rounds: {mm} client-rounds "
          f"(0 once client c is pinned to worker c % 2; the verifier above bounds any "
          f"cross-GPU deviation at proj.verify.CACHE_TOL_ROWS rows)")

# Calibration, from THIS session's rounds only (the CSV would mix in imported rounds).
sec = [float(r["seconds"]) for r in hist]
overhead = (time.monotonic() - T0) - sum(sec)
print(f"\nbackend  : train {CFG.get('backend', '?')} | eval {CFG.get('backend_eval', '?')}")
print(f"session  : {len(hist)} round(s) here | startup+prepack+compile {overhead/60:.1f} min"
      f" | verify and W&B are outside this figure")
if len(sec) < 2:
    print("timing   : need 2 completed rounds to separate startup from steady state; "
          f"got {len(sec)}. No projection.")
else:
    steady = sum(sec[1:]) / len(sec[1:])
    vt = max(float(r.get("vram_train_gb", 0) or 0) for r in hist)
    ve = max(float(r.get("vram_eval_gb", 0) or 0) for r in hist)
    tr = sum(float(r["train_sec"]) for r in hist[1:]) / len(hist[1:])
    ev = sum(float(r["eval_sec"]) for r in hist[1:]) / len(hist[1:])
    print(f"timing   : rounds {[round(x) for x in sec[-3:]]}s | steady {steady:.0f}s/round "
          f"= train {tr:.0f}s + eval {ev:.0f}s (evaluated {hist[-1]['evaluated']} clients) + commit")
    print(f"VRAM     : train {vt:.2f} GiB/GPU | eval {ve:.2f} GiB/GPU (of 16)")
    print(f"projected: {CFG['rounds']} rounds = "
          f"{(steady*CFG['rounds'] + overhead)/3600:.2f} h "
          f"({(steady*CFG['rounds'])/3600:.2f} h of rounds + {overhead/3600:.2f} h startup) "
          f"— full-eval rounds cost more than the steady figure")
if run is not None: run.finish()
assert ok, "artifact verification FAILED — see the FAIL lines above"
